
#Stage 1 — Visual DL Classifier
#image → {سباكة, كهرباء, نجارة, نقاشة, irrelevant}


In [1]:
!pip install -q ultralytics timm scikit-learn pandas matplotlib pillow tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 70.5 MB/s eta 0:00:00


In [2]:

import os
import random
import shutil
import json
import time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

from ultralytics import YOLO

ImageFile.LOAD_TRUNCATED_IMAGES = True

# ============================================================
# 0. CONFIG
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# عدلي المسار ده لو الداتا عندك في مكان مختلف
SOURCE_ROOT = Path("/content/drive/MyDrive/Fake Image Detection")



WORK_DIR = Path("/content/drive/MyDrive/stage1_visual_dl")
DATASET_DIR = WORK_DIR / "visual_dataset_5class"
RESULTS_DIR = WORK_DIR / "results"
BEST_DIR = WORK_DIR / "best_model"

for d in [WORK_DIR, DATASET_DIR, RESULTS_DIR, BEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

CLASS_MAP = {
    "سباكه": "plumbing",
    "سباكة": "plumbing",
    "كهربا": "electricity",
    "كهرباء": "electricity",
    "نجاره": "carpentry",
    "نجارة": "carpentry",
    "نقاشه": "painting",
    "نقاشة": "painting",
    "irrelevant": "irrelevant",
}

CLASS_NAMES = ["plumbing", "electricity", "carpentry", "painting", "irrelevant"]

ARABIC_NAMES = {
    "plumbing": "سباكة",
    "electricity": "كهرباء",
    "carpentry": "نجارة",
    "painting": "نقاشة",
    "irrelevant": "irrelevant",
}

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

YOLO_EPOCHS = 40
TORCH_EPOCHS = 15
PATIENCE = 8

print("SOURCE_ROOT:", SOURCE_ROOT)
print("Exists:", SOURCE_ROOT.exists())

if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"Dataset root not found: {SOURCE_ROOT}")

# ============================================================
# 1. COLLECT IMAGES FROM CURRENT FOLDER STRUCTURE
# ============================================================

def collect_images_from_current_structure(source_root):
    rows = []

    relevant_root = source_root / "relevant"
    irrelevant_root = source_root / "irrelevant"

    if relevant_root.exists():
        for folder in relevant_root.iterdir():
            if not folder.is_dir():
                continue

            raw_name = folder.name.strip()
            class_name = CLASS_MAP.get(raw_name)

            if class_name is None:
                print("⚠️ Unknown relevant folder ignored:", folder)
                continue

            for p in folder.rglob("*"):
                if p.is_file() and p.suffix.lower() in VALID_EXTS:
                    rows.append({
                        "path": str(p),
                        "class_name": class_name,
                        "source_folder": str(folder)
                    })

    if irrelevant_root.exists():
        for p in irrelevant_root.rglob("*"):
            if p.is_file() and p.suffix.lower() in VALID_EXTS:
                rows.append({
                    "path": str(p),
                    "class_name": "irrelevant",
                    "source_folder": str(irrelevant_root)
                })

    df = pd.DataFrame(rows)

    if len(df) == 0:
        raise ValueError("No images found. Check SOURCE_ROOT and folder structure.")

    df = df.drop_duplicates(subset=["path"]).reset_index(drop=True)

    return df

df = collect_images_from_current_structure(SOURCE_ROOT)

print("\n===== DATASET SUMMARY =====")
print("Total images:", len(df))
print(df["class_name"].value_counts())

missing_classes = set(CLASS_NAMES) - set(df["class_name"].unique())
if missing_classes:
    print("⚠️ Missing classes:", missing_classes)

# ============================================================
# 2. VERIFY IMAGES AND REMOVE BROKEN FILES
# ============================================================

def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

print("\nChecking image files...")
valid_flags = []
for p in tqdm(df["path"].tolist()):
    valid_flags.append(is_valid_image(p))

df["valid"] = valid_flags
bad_df = df[~df["valid"]].copy()
df = df[df["valid"]].drop(columns=["valid"]).reset_index(drop=True)

print("Valid images:", len(df))
print("Broken/invalid images:", len(bad_df))

if len(bad_df) > 0:
    bad_df.to_csv(RESULTS_DIR / "broken_images.csv", index=False)
    print("Saved broken image list:", RESULTS_DIR / "broken_images.csv")

print("\nClass distribution after cleaning:")
print(df["class_name"].value_counts())

# ============================================================
# 3. TRAIN / VAL / TEST SPLIT
# ============================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["class_name"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["class_name"]
)

split_dfs = {
    "train": train_df.reset_index(drop=True),
    "val": val_df.reset_index(drop=True),
    "test": test_df.reset_index(drop=True),
}

print("\n===== SPLIT SUMMARY =====")
for split, sdf in split_dfs.items():
    print(f"\n{split}: {len(sdf)}")
    print(sdf["class_name"].value_counts())

# ============================================================
# 4. CREATE YOLO/TORCH CLASSIFICATION DATASET STRUCTURE
# ============================================================

def safe_copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        shutil.copy2(src, dst)

print("\nCreating classification dataset folders...")

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

for split, sdf in split_dfs.items():
    for class_name in CLASS_NAMES:
        (DATASET_DIR / split / class_name).mkdir(parents=True, exist_ok=True)

    for idx, row in tqdm(sdf.iterrows(), total=len(sdf), desc=f"Copying {split}"):
        src = Path(row["path"])
        class_name = row["class_name"]
        dst_name = f"{idx:06d}_{src.stem}{src.suffix.lower()}"
        dst = DATASET_DIR / split / class_name / dst_name
        safe_copy(src, dst)

metadata = {
    "class_names": CLASS_NAMES,
    "arabic_names": ARABIC_NAMES,
    "source_root": str(SOURCE_ROOT),
    "dataset_dir": str(DATASET_DIR),
    "splits": {k: len(v) for k, v in split_dfs.items()}
}

with open(RESULTS_DIR / "dataset_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("\n✅ Dataset prepared at:", DATASET_DIR)

# ============================================================
# 5. EVALUATION HELPERS
# ============================================================

def compute_metrics(y_true, y_pred, labels):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    macro_precision = precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0
    )

    irrelevant_recall = report_dict.get("irrelevant", {}).get("recall", 0.0)

    irrelevant_total = sum([1 for y in y_true if y == "irrelevant"])
    irrelevant_false_accept = sum([
        1 for yt, yp in zip(y_true, y_pred)
        if yt == "irrelevant" and yp != "irrelevant"
    ])

    false_accept_rate = irrelevant_false_accept / max(irrelevant_total, 1)

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "irrelevant_recall": irrelevant_recall,
        "irrelevant_false_accept_rate": false_accept_rate,
        "classification_report": report_dict,
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist()
    }

def save_eval_outputs(model_name, y_true, y_pred, metrics):
    model_dir = RESULTS_DIR / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    pred_df = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred
    })
    pred_df.to_csv(model_dir / "test_predictions.csv", index=False)

    with open(model_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    report_df = pd.DataFrame(metrics["classification_report"]).T
    report_df.to_csv(model_dir / "classification_report.csv")

    cm_df = pd.DataFrame(
        metrics["confusion_matrix"],
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    )
    cm_df.to_csv(model_dir / "confusion_matrix.csv")

# ============================================================
# 6. YOLO-CLS TRAINING + EVALUATION
# ============================================================

def train_eval_yolo(model_ckpt):
    model_name = model_ckpt.replace(".pt", "").replace("-", "_")
    print("\n" + "=" * 80)
    print("TRAINING YOLO:", model_ckpt)
    print("=" * 80)

    model = YOLO(model_ckpt)

    train_result = model.train(
        data=str(DATASET_DIR),
        epochs=YOLO_EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        patience=PATIENCE,
        seed=SEED,
        project=str(RESULTS_DIR / "yolo_runs"),
        name=model_name,
        exist_ok=True,
        verbose=False
    )

    best_weights = RESULTS_DIR / "yolo_runs" / model_name / "weights" / "best.pt"

    if not best_weights.exists():
        best_weights = RESULTS_DIR / "yolo_runs" / model_name / "weights" / "last.pt"

    best_model = YOLO(str(best_weights))

    test_image_paths = []
    y_true = []

    for class_name in CLASS_NAMES:
        for p in sorted((DATASET_DIR / "test" / class_name).glob("*")):
            if p.suffix.lower() in VALID_EXTS:
                test_image_paths.append(str(p))
                y_true.append(class_name)

    y_pred = []

    for p in tqdm(test_image_paths, desc=f"Predicting {model_name}"):
        r = best_model(p, verbose=False)[0]
        pred_idx = int(r.probs.top1)
        pred_name = r.names[pred_idx]
        y_pred.append(pred_name)

    metrics = compute_metrics(y_true, y_pred, CLASS_NAMES)
    save_eval_outputs(model_name, y_true, y_pred, metrics)

    row = {
        "model": model_name,
        "type": "YOLO-cls",
        "weights_path": str(best_weights),
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
        "macro_precision": metrics["macro_precision"],
        "macro_recall": metrics["macro_recall"],
        "irrelevant_recall": metrics["irrelevant_recall"],
        "irrelevant_false_accept_rate": metrics["irrelevant_false_accept_rate"],
    }

    print("\nYOLO result:")
    for k, v in row.items():
        print(k, ":", v)

    return row

# ============================================================
# 7. TORCHVISION/TIMM TRAINING + EVALUATION
# ============================================================

class EarlyStopper:
    def __init__(self, patience=5, mode="max"):
        self.patience = patience
        self.mode = mode
        self.best = None
        self.counter = 0

    def step(self, value):
        if self.best is None:
            self.best = value
            self.counter = 0
            return False, True

        improved = value > self.best if self.mode == "max" else value < self.best

        if improved:
            self.best = value
            self.counter = 0
            return False, True

        self.counter += 1
        return self.counter >= self.patience, False

def make_dataloaders():
    train_tfms = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=8),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    eval_tfms = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    train_ds = datasets.ImageFolder(DATASET_DIR / "train", transform=train_tfms)
    val_ds = datasets.ImageFolder(DATASET_DIR / "val", transform=eval_tfms)
    test_ds = datasets.ImageFolder(DATASET_DIR / "test", transform=eval_tfms)

    print("Torch class_to_idx:", train_ds.class_to_idx)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    idx_to_class = {v: k for k, v in train_ds.class_to_idx.items()}

    return train_loader, val_loader, test_loader, idx_to_class, train_ds.class_to_idx

def evaluate_torch_model(model, loader, idx_to_class):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(images)
            preds = torch.argmax(logits, dim=1)

            y_true.extend([idx_to_class[int(x)] for x in labels.cpu().numpy()])
            y_pred.extend([idx_to_class[int(x)] for x in preds.cpu().numpy()])

    metrics = compute_metrics(y_true, y_pred, CLASS_NAMES)

    return metrics, y_true, y_pred

def train_eval_timm(model_arch):
    model_name = model_arch.replace("/", "_").replace("-", "_")
    print("\n" + "=" * 80)
    print("TRAINING TIMM:", model_arch)
    print("=" * 80)

    train_loader, val_loader, test_loader, idx_to_class, class_to_idx = make_dataloaders()

    model = timm.create_model(
        model_arch,
        pretrained=True,
        num_classes=len(CLASS_NAMES)
    ).to(DEVICE)

    class_counts = []
    train_targets = [y for _, y in train_loader.dataset.samples]
    for i in range(len(CLASS_NAMES)):
        class_counts.append(train_targets.count(i))

    class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float32)
    class_weights = class_weights / class_weights.sum() * len(CLASS_NAMES)
    class_weights = class_weights.to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TORCH_EPOCHS)

    model_dir = RESULTS_DIR / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    best_path = model_dir / "best_model.pt"

    stopper = EarlyStopper(patience=PATIENCE, mode="max")
    history = []

    for epoch in range(1, TORCH_EPOCHS + 1):
        model.train()
        train_loss = 0.0

        for images, labels in tqdm(train_loader, desc=f"{model_name} epoch {epoch}/{TORCH_EPOCHS}", leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        scheduler.step()

        train_loss /= len(train_loader.dataset)

        val_metrics, _, _ = evaluate_torch_model(model, val_loader, idx_to_class)
        val_score = 0.70 * val_metrics["macro_f1"] + 0.30 * val_metrics["irrelevant_recall"]

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_irrelevant_recall": val_metrics["irrelevant_recall"],
            "val_score": val_score
        })

        print(
            f"Epoch {epoch:02d} | "
            f"loss={train_loss:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
            f"val_irrel_recall={val_metrics['irrelevant_recall']:.4f} | "
            f"score={val_score:.4f}"
        )

        should_stop, improved = stopper.step(val_score)

        if improved:
            torch.save({
                "model_arch": model_arch,
                "state_dict": model.state_dict(),
                "class_names": CLASS_NAMES,
                "class_to_idx": class_to_idx,
                "idx_to_class": idx_to_class,
                "img_size": IMG_SIZE
            }, best_path)

        if should_stop:
            print("Early stopping.")
            break

    pd.DataFrame(history).to_csv(model_dir / "training_history.csv", index=False)

    checkpoint = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["state_dict"])

    test_metrics, y_true, y_pred = evaluate_torch_model(model, test_loader, idx_to_class)
    save_eval_outputs(model_name, y_true, y_pred, test_metrics)

    row = {
        "model": model_name,
        "type": "timm",
        "weights_path": str(best_path),
        "accuracy": test_metrics["accuracy"],
        "macro_f1": test_metrics["macro_f1"],
        "weighted_f1": test_metrics["weighted_f1"],
        "macro_precision": test_metrics["macro_precision"],
        "macro_recall": test_metrics["macro_recall"],
        "irrelevant_recall": test_metrics["irrelevant_recall"],
        "irrelevant_false_accept_rate": test_metrics["irrelevant_false_accept_rate"],
    }

    print("\nTIMM result:")
    for k, v in row.items():
        print(k, ":", v)

    return row

# ============================================================
# 8. RUN EXPERIMENTS
# ============================================================

all_results = []

RUN_YOLO = True
RUN_TIMM = True

YOLO_MODELS = [
    "yolo11n-cls.pt",
    "yolo11s-cls.pt",
]

TIMM_MODELS = [
    "efficientnet_b0",
    "mobilenetv3_large_100",
    "resnet50",
]

if RUN_YOLO:
    for ckpt in YOLO_MODELS:
        try:
            row = train_eval_yolo(ckpt)
            all_results.append(row)
        except Exception as e:
            print(f"❌ YOLO model failed: {ckpt}")
            print(e)

if RUN_TIMM:
    for arch in TIMM_MODELS:
        try:
            row = train_eval_timm(arch)
            all_results.append(row)
        except Exception as e:
            print(f"❌ TIMM model failed: {arch}")
            print(e)

# ============================================================
# 9. RESULTS TABLE + BEST MODEL SELECTION
# ============================================================

results_df = pd.DataFrame(all_results)

if len(results_df) == 0:
    raise RuntimeError("No models finished successfully.")

results_df["selection_score"] = (
    0.70 * results_df["macro_f1"]
    +
    0.30 * results_df["irrelevant_recall"]
)

results_df = results_df.sort_values(
    by=["selection_score", "macro_f1", "irrelevant_recall"],
    ascending=False
).reset_index(drop=True)

results_path = RESULTS_DIR / "stage1_visual_model_comparison.csv"
results_df.to_csv(results_path, index=False)

print("\n" + "=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)
display(results_df)

best_row = results_df.iloc[0].to_dict()

print("\n🏆 BEST MODEL")
for k, v in best_row.items():
    print(k, ":", v)

with open(BEST_DIR / "best_model_info.json", "w", encoding="utf-8") as f:
    json.dump(best_row, f, ensure_ascii=False, indent=2)

# Copy best weights
best_weights_src = Path(best_row["weights_path"])
best_weights_dst = BEST_DIR / best_weights_src.name

if best_weights_src.exists():
    shutil.copy2(best_weights_src, best_weights_dst)
    print("\n✅ Best weights copied to:", best_weights_dst)

print("\n✅ Results saved to:", results_path)
print("✅ Best model info saved to:", BEST_DIR / "best_model_info.json")

# ============================================================
# 10. QUICK PREDICTION FUNCTION FOR BEST MODEL
# ============================================================

def predict_stage1_visual_best(image_path):
    image_path = str(image_path)
    model_type = best_row["type"]
    weights_path = best_row["weights_path"]

    if model_type == "YOLO-cls":
        model = YOLO(weights_path)
        r = model(image_path, verbose=False)[0]
        pred_idx = int(r.probs.top1)
        pred_class = r.names[pred_idx]
        confidence = float(r.probs.top1conf)

        return {
            "stage": "Stage 1 Visual DL Classifier",
            "image_path": image_path,
            "model": best_row["model"],
            "image_class": pred_class,
            "image_class_ar": ARABIC_NAMES[pred_class],
            "confidence": confidence,
            "decision": "IRRELEVANT" if pred_class == "irrelevant" else "RELEVANT"
        }

    else:
        checkpoint = torch.load(weights_path, map_location=DEVICE)
        model_arch = checkpoint["model_arch"]
        idx_to_class = checkpoint["idx_to_class"]

        model = timm.create_model(
            model_arch,
            pretrained=False,
            num_classes=len(CLASS_NAMES)
        ).to(DEVICE)

        model.load_state_dict(checkpoint["state_dict"])
        model.eval()

        tfm = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

        img = Image.open(image_path).convert("RGB")
        x = tfm(img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[0]
            pred_idx = int(torch.argmax(probs).item())
            confidence = float(probs[pred_idx].item())

        pred_class = idx_to_class[pred_idx]

        return {
            "stage": "Stage 1 Visual DL Classifier",
            "image_path": image_path,
            "model": best_row["model"],
            "image_class": pred_class,
            "image_class_ar": ARABIC_NAMES[pred_class],
            "confidence": confidence,
            "decision": "IRRELEVANT" if pred_class == "irrelevant" else "RELEVANT"
        }

print("\n✅ Ready. Use:")
print("predict_stage1_visual_best('/content/WhatsApp Image 2026-06-07 at 10.52.23 PM.jpeg')")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
DEVICE: cuda
SOURCE_ROOT: /content/drive/MyDrive/Fake Image Detection
Exists: True

===== DATASET SUMMARY =====
Total images: 4958
class_name
irrelevant     3021
painting        712
electricity     457
carpentry       426
plumbing        342
Name: count, dtype: int64

Checking image files...


100%|██████████| 4958/4958 [59:30<00:00,  1.39it/s]


Valid images: 4958
Broken/invalid images: 0

Class distribution after cleaning:
class_name
irrelevant     3021
painting        712
electricity     457
carpentry       426
plumbing        342
Name: count, dtype: int64

===== SPLIT SUMMARY =====

train: 3470
class_name
irrelevant     2114
painting        498
electricity     320
carpentry       298
plumbing        240
Name: count, dtype: int64

val: 744
class_name
irrelevant     453
painting       107
electricity     69
carpentry       64
plumbing        51
Name: count, dtype: int64

test: 744
class_name
irrelevant     454
painting       107
electricity     68
carpentry       64
plumbing        51
Name: count, dtype: int64

Creating classification dataset folders...


Copying test: 100%|██████████| 744/744 [00:16<00:00, 45.42it/s]



✅ Dataset prepared at: /content/drive/MyDrive/stage1_visual_dl/visual_dataset_5class

TRAINING YOLO: yolo11n-cls.pt
Ultralytics 8.4.64 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/stage1_visual_dl/visual_dataset_5class, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, 

...

Epoch 13 | loss=0.0131 | val_acc=0.9583 | val_macro_f1=0.9217 | val_irrel_recall=0.9890 | score=0.9419


Epoch 14 | loss=0.0161 | val_acc=0.9583 | val_macro_f1=0.9195 | val_irrel_recall=0.9890 | score=0.9403


Epoch 15 | loss=0.0156 | val_acc=0.9597 | val_macro_f1=0.9245 | val_irrel_recall=0.9890 | score=0.9439
Early stopping.

TIMM result:
model : resnet50
type : timm
weights_path : /content/drive/MyDrive/stage1_visual_dl/results/resnet50/best_model.pt
accuracy : 0.9475806451612904
macro_f1 : 0.9018185888700245
weighted_f1 : 0.9479011038884495
macro_precision : 0.8995636228807612
macro_recall : 0.9072599736506237
irrelevant_recall : 0.9889867841409692
irrelevant_false_accept_rate : 0.011013215859030838

FINAL MODEL COMPARISON


,model,type,weights_path,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,irrelevant_recall,irrelevant_false_accept_rate,selection_score
0,resnet50,timm,/content/drive/MyDrive/stage1_visual_dl/result...,0.947581,0.901819,0.947901,0.899564,0.907260,0.988987,0.011013,0.927969
1,mobilenetv3_large_100,timm,/content/drive/MyDrive/stage1_visual_dl/result...,0.944892,0.896069,0.945183,0.894378,0.898598,0.984581,0.015419,0.922622
2,efficientnet_b0,timm,/content/drive/MyDrive/stage1_visual_dl/result...,0.932796,0.875929,0.933356,0.869320,0.887401,0.982379,0.017621,0.907864
3,yolo11n_cls,YOLO-cls,/content/drive/MyDrive/stage1_visual_dl/result...,0.908602,0.834479,0.906451,0.864382,0.815039,0.982379,0.017621,0.878849
4,yolo11s_cls,YOLO-cls,/content/drive/MyDrive/stage1_visual_dl/result...,0.895161,0.818478,0.893193,0.836581,0.803023,0.975771,0.024229,0.865666



🏆 BEST MODEL
model : resnet50
type : timm
weights_path : /content/drive/MyDrive/stage1_visual_dl/results/resnet50/best_model.pt
accuracy : 0.9475806451612904
macro_f1 : 0.9018185888700245
weighted_f1 : 0.9479011038884495
macro_precision : 0.8995636228807612
macro_recall : 0.9072599736506237
irrelevant_recall : 0.9889867841409692
irrelevant_false_accept_rate : 0.011013215859030838
selection_score : 0.9279690474513078

✅ Best weights copied to: /content/drive/MyDrive/stage1_visual_dl/best_model/best_model.pt

✅ Results saved to: /content/drive/MyDrive/stage1_visual_dl/results/stage1_visual_model_comparison.csv
✅ Best model info saved to: /content/drive/MyDrive/stage1_visual_dl/best_model/best_model_info.json

✅ Ready. Use:
predict_stage1_visual_best('/content/WhatsApp Image 2026-06-07 at 10.52.23 PM.jpeg')


In [3]:
!pip install -q gradio timm ultralytics

In [4]:
# ============================================================
# Gradio Interface for Stage 1 Visual DL Classifier
# Works with the best model selected from the previous training cell
# ============================================================



import json
import tempfile
from pathlib import Path

import torch
import timm
import pandas as pd
import gradio as gr
from PIL import Image
from torchvision import transforms
from ultralytics import YOLO

# ============================================================
# 1. Paths
# ============================================================

STAGE1_DIR = Path("/content/drive/MyDrive/stage1_visual_dl")
BEST_DIR = STAGE1_DIR / "best_model"
BEST_INFO_PATH = BEST_DIR / "best_model_info.json"

if not BEST_INFO_PATH.exists():
    raise FileNotFoundError(f"Best model info not found: {BEST_INFO_PATH}")

with open(BEST_INFO_PATH, "r", encoding="utf-8") as f:
    best_info = json.load(f)

print("Loaded best model info:")
print(json.dumps(best_info, indent=2, ensure_ascii=False))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

CLASS_NAMES = ["plumbing", "electricity", "carpentry", "painting", "irrelevant"]

ARABIC_NAMES = {
    "plumbing": "سباكة",
    "electricity": "كهرباء",
    "carpentry": "نجارة",
    "painting": "نقاشة",
    "irrelevant": "irrelevant"
}

CLASS_DESCRIPTIONS = {
    "plumbing": "سباكة / Plumbing",
    "electricity": "كهرباء / Electricity",
    "carpentry": "نجارة / Carpentry",
    "painting": "نقاشة / Painting",
    "irrelevant": "Irrelevant / صورة غير متعلقة بالصيانة"
}

IMG_SIZE = 224

# ============================================================
# 2. Load Best Model
# ============================================================

MODEL_TYPE = best_info["type"]
WEIGHTS_PATH = Path(best_info["weights_path"])

if not WEIGHTS_PATH.exists():
    # fallback if copied inside best_model folder
    candidate = BEST_DIR / "best_model.pt"
    if candidate.exists():
        WEIGHTS_PATH = candidate
    else:
        raise FileNotFoundError(f"Best weights not found: {best_info['weights_path']}")

print("MODEL_TYPE:", MODEL_TYPE)
print("WEIGHTS_PATH:", WEIGHTS_PATH)

if MODEL_TYPE == "YOLO-cls":
    stage1_model = YOLO(str(WEIGHTS_PATH))
    torch_model = None
    idx_to_class = None

else:
    checkpoint = torch.load(WEIGHTS_PATH, map_location=DEVICE)

    model_arch = checkpoint["model_arch"]
    idx_to_class = checkpoint["idx_to_class"]

    # Sometimes dict keys are saved as strings; normalize them
    idx_to_class = {int(k): v for k, v in idx_to_class.items()}

    stage1_model = timm.create_model(
        model_arch,
        pretrained=False,
        num_classes=len(CLASS_NAMES)
    ).to(DEVICE)

    stage1_model.load_state_dict(checkpoint["state_dict"])
    stage1_model.eval()

    torch_model = stage1_model

    eval_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

print("✅ Model loaded successfully.")

# ============================================================
# 3. Prediction Function
# ============================================================

def predict_stage1_gradio(image):
    if image is None:
        return (
            "No image uploaded.",
            "N/A",
            0.0,
            pd.DataFrame(columns=["class", "arabic_name", "probability"]),
            {}
        )

    image = image.convert("RGB")

    if MODEL_TYPE == "YOLO-cls":
        with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
            image.save(tmp.name)
            tmp_path = tmp.name

        result = stage1_model(tmp_path, verbose=False)[0]

        probs_tensor = result.probs.data.detach().cpu()
        probs = probs_tensor.numpy()

        names = result.names
        pred_idx = int(result.probs.top1)
        pred_class = names[pred_idx]
        confidence = float(result.probs.top1conf)

        prob_rows = []
        for i, p in enumerate(probs):
            cls = names[i]
            prob_rows.append({
                "class": CLASS_DESCRIPTIONS.get(cls, cls),
                "arabic_name": ARABIC_NAMES.get(cls, cls),
                "probability": float(p)
            })

    else:
        x = eval_transform(image).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logits = torch_model(x)
            probs_tensor = torch.softmax(logits, dim=1)[0].detach().cpu()
            probs = probs_tensor.numpy()

        pred_idx = int(probs_tensor.argmax().item())
        pred_class = idx_to_class[pred_idx]
        confidence = float(probs[pred_idx])

        prob_rows = []
        for i, p in enumerate(probs):
            cls = idx_to_class[i]
            prob_rows.append({
                "class": CLASS_DESCRIPTIONS.get(cls, cls),
                "arabic_name": ARABIC_NAMES.get(cls, cls),
                "probability": float(p)
            })

    prob_df = pd.DataFrame(prob_rows)
    prob_df = prob_df.sort_values("probability", ascending=False).reset_index(drop=True)

    if pred_class == "irrelevant":
        decision = "IRRELEVANT → MISMATCH"
        decision_ar = "الصورة غير متعلقة بالصيانة → MISMATCH"
    else:
        decision = "RELEVANT"
        decision_ar = f"الصورة متعلقة بالصيانة، الفئة المتوقعة: {ARABIC_NAMES[pred_class]}"

    summary = {
        "stage": "Stage 1 Visual DL Classifier",
        "model": best_info["model"],
        "model_type": MODEL_TYPE,
        "predicted_class": pred_class,
        "predicted_class_ar": ARABIC_NAMES[pred_class],
        "confidence": confidence,
        "decision": decision,
        "decision_ar": decision_ar
    }

    label_text = f"""
### Prediction Result

**Model:** {best_info["model"]}
**Predicted class:** {CLASS_DESCRIPTIONS[pred_class]}
**Confidence:** {confidence:.4f}
**Decision:** {decision}

**Arabic explanation:**
{decision_ar}
"""

    return (
        label_text,
        CLASS_DESCRIPTIONS[pred_class],
        confidence,
        prob_df,
        summary
    )

# ============================================================
# 4. Gradio App
# ============================================================

demo = gr.Interface(
    fn=predict_stage1_gradio,
    inputs=gr.Image(type="pil", label="Upload maintenance / irrelevant image"),
    outputs=[
        gr.Markdown(label="Result Summary"),
        gr.Textbox(label="Predicted Class"),
        gr.Number(label="Confidence"),
        gr.Dataframe(label="Class Probabilities"),
        gr.JSON(label="Raw Output")
    ],
    title="Stage 1 Visual DL Classifier — Maintenance Image Relevance & Category",
    description="""
Upload an image to classify it as:
سباكة، كهرباء، نجارة، نقاشة، or irrelevant.

If the image is classified as irrelevant, the system should reject it before the full matching pipeline.
""",
    allow_flagging="never"
)

demo.launch(share=True, debug=True)


Loaded best model info:
{
  "model": "resnet50",
  "type": "timm",
  "weights_path": "/content/drive/MyDrive/stage1_visual_dl/results/resnet50/best_model.pt",
  "accuracy": 0.9475806451612904,
  "macro_f1": 0.9018185888700245,
  "weighted_f1": 0.9479011038884495,
  "macro_precision": 0.8995636228807612,
  "macro_recall": 0.9072599736506237,
  "irrelevant_recall": 0.9889867841409692,
  "irrelevant_false_accept_rate": 0.011013215859030838,
  "selection_score": 0.9279690474513078
}
DEVICE: cuda
MODEL_TYPE: timm
WEIGHTS_PATH: /content/drive/MyDrive/stage1_visual_dl/results/resnet50/best_model.pt
✅ Model loaded successfully.


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ac9578204263313f0e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ac9578204263313f0e.gradio.live


#Stage 2 — Arabic Text Category Classifier
#arabic_description → {سباكة, كهرباء, نجارة, نقاشة}
#Hybrid: ML text classifier + keyword detector

In [5]:
!pip install -q scikit-learn pandas numpy joblib gradio xgboost
!pip install -q sentence-transformers scikit-learn pandas numpy scipy joblib gradio

ERROR: Operation cancelled by user


In [6]:
# ============================================================
# Stage 2 v2 — Improved Arabic Text Classifier
# MiniLM + TF-IDF + Keyword Features + Hybrid Fusion
# Classes:
# سباكة / كهرباء / نجارة / نقاشة / غير متعلق بالصيانة
# ============================================================

import re
import json
import joblib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sentence_transformers import SentenceTransformer

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

import gradio as gr

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# ============================================================
# 0. Paths
# ============================================================

WORK_DIR = Path("/content/drive/MyDrive/stage2_final_text_v2")
ARTIFACTS_DIR = WORK_DIR / "artifacts"
RESULTS_DIR = WORK_DIR / "results"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

POSSIBLE_DATASET_PATHS = [
    Path("/content/stage2_unified_clean_dataset.csv"),
    Path("/content/drive/MyDrive/stage2_unified_clean_dataset.csv"),
    Path("/content/drive/MyDrive/Fake Image Detection/stage2_unified_clean_dataset.csv"),
]

DATASET_PATH = None
for p in POSSIBLE_DATASET_PATHS:
    if p.exists():
        DATASET_PATH = p
        break

if DATASET_PATH is None:
    raise FileNotFoundError("stage2_unified_clean_dataset.csv not found.")

print("✅ Using dataset:", DATASET_PATH)

CATEGORIES = ["سباكة", "كهرباء", "نجارة", "نقاشة", "غير متعلق بالصيانة"]
POSITIVE_CATEGORIES = ["سباكة", "كهرباء", "نجارة", "نقاشة"]

# ============================================================
# 1. Arabic normalization
# ============================================================

def normalize_arabic(text):
    text = "" if pd.isna(text) else str(text)
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ة", "ه").replace("ى", "ي")
    text = text.replace("ؤ", "و").replace("ئ", "ي")
    text = text.replace("گ", "ك")
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)
    text = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ============================================================
# 2. Stronger keyword dictionary
# ============================================================

CATEGORY_KEYWORDS = {
    "سباكة": [
        "سباكه", "سباكة", "سباك", "مواسير", "ماسوره", "ماسورة",
        "مياه", "ميه", "ماء", "حنفيه", "حنفية", "صنبور", "خلاط",
        "محبس", "حوض", "صرف", "بلاعه", "بلاعة", "مجاري", "تسليك",
        "انسداد", "مسدود", "مرحاض", "تواليت", "قاعدة", "قاعده",
        "حمام", "سيفون", "شطاف", "بانيو", "دش", "تسريب", "تسرب",
        "بيسرب", "بتسرب", "بتنقط", "بينقط", "تنقيط", "بلل",
        "رطوبه", "رطوبة", "طفح", "بيطفح", "خزان", "خرطوم",
        "صرف صحي", "مياه الصرف", "مواسير المياه", "تحت الحوض"
    ],
    "كهرباء": [
        "كهرباء", "كهربا", "كهربائي", "كهربائى", "نور", "انوار",
        "أنوار", "اضاءه", "اضاءة", "إضاءة", "لمبه", "لمبة", "لمبات",
        "مصباح", "كشاف", "فيشه", "فيشة", "بريزه", "بريزة", "مقبس",
        "مشترك", "سلك", "سلوك", "اسلاك", "أسلاك", "كابل", "كابلات",
        "شرار", "شرز", "ماس", "قفله", "قفلة", "شورت", "فاصل",
        "فاصله", "فاصلة", "بيفصل", "بتفصل", "بيقطع", "بتقطع",
        "قاطع", "لوحه", "لوحة", "عداد", "مفتاح", "زر", "محروق",
        "شياط", "سخونه", "سخونة", "مروحه", "مروحة", "شفاط", "جرس",
        "قاطع كهرباء", "لوحة كهرباء", "مفتاح نور", "ريحة شياط"
    ],
    "نجارة": [
        "نجاره", "نجارة", "نجار", "خشب", "خشبي", "خشبى", "باب",
        "ابواب", "أبواب", "شباك", "شبابيك", "نافذه", "نافذة",
        "درفه", "درفة", "ضلفه", "ضلفة", "دولاب", "مطبخ", "درج",
        "ادراج", "أدراج", "رف", "ارفف", "أرفف", "مفصله", "مفصلة",
        "مفصلات", "كالون", "مقبض", "اكره", "أكرة", "اكرة", "مكسور",
        "كسر", "مخلوع", "مفكوك", "متفكك", "بيحك", "بيعلق",
        "مش بيقفل", "مش بيتقفل", "لا يغلق", "مش بيفتح", "اطار",
        "إطار", "برواز", "اثاث", "أثاث", "ترابيزة", "كرسي",
        "باب خشب", "درفة مكسورة", "خشب مكسور", "مقبض الباب"
    ],
    "نقاشة": [
        "نقاشه", "نقاشة", "نقاش", "دهان", "دهانات", "طلاء",
        "بويه", "بوية", "لون", "تشطيب", "محاره", "محارة",
        "حائط", "حيطه", "حيطة", "جدار", "جدران", "حوايط",
        "سقف", "اسقف", "أسقف", "شرخ", "شروخ", "تشققات", "تشقق",
        "مشققه", "مشققة", "رطوبه", "رطوبة", "نشع", "بقع", "بقعه",
        "بقعة", "تقشير", "مقشر", "مقشره", "مقشرة", "متقشر",
        "متقشرة", "الدهان واقع", "الطلاء واقع", "تساقط", "تلف دهان",
        "بهتان", "عفن", "متعفن", "اثار مياه", "آثار مياه",
        "دهان مقشر", "حائط مقشر", "سقف متشقق", "بقع سقف"
    ],
}

NORMALIZED_KEYWORDS = {
    cat: sorted(set(normalize_arabic(k) for k in kws if normalize_arabic(k)))
    for cat, kws in CATEGORY_KEYWORDS.items()
}

def keyword_detector(text):
    norm = normalize_arabic(text)

    scores = {}
    hits = {}

    for cat, kws in NORMALIZED_KEYWORDS.items():
        score = 0.0
        cat_hits = []

        for kw in kws:
            if kw and kw in norm:
                cat_hits.append(kw)

                if len(kw.split()) >= 2:
                    score += 2.5
                elif len(kw) >= 6:
                    score += 1.5
                elif len(kw) >= 4:
                    score += 1.2
                else:
                    score += 1.0

        scores[cat] = score
        hits[cat] = cat_hits

    best_cat = max(scores, key=scores.get)
    best_score = scores[best_cat]
    total_score = sum(scores.values())

    sorted_scores = sorted(scores.values(), reverse=True)
    second_score = sorted_scores[1] if len(sorted_scores) > 1 else 0.0
    margin = best_score - second_score

    if total_score == 0:
        return {
            "decision": "NO_KEYWORD",
            "category": None,
            "confidence": 0.0,
            "scores": scores,
            "hits": hits,
            "best_score": 0.0,
            "second_score": 0.0,
            "margin": 0.0,
            "total_score": 0.0
        }

    confidence = best_score / total_score

    if best_score >= 1.0 and margin >= 0.5:
        decision = "KEYWORD_MATCH"
    else:
        decision = "KEYWORD_AMBIGUOUS"

    return {
        "decision": decision,
        "category": best_cat,
        "confidence": float(confidence),
        "scores": scores,
        "hits": hits,
        "best_score": float(best_score),
        "second_score": float(second_score),
        "margin": float(margin),
        "total_score": float(total_score)
    }

def keyword_feature_vector(text):
    r = keyword_detector(text)
    scores = r["scores"]

    raw_scores = [scores[c] for c in POSITIVE_CATEGORIES]
    total = sum(raw_scores)

    if total > 0:
        norm_scores = [s / total for s in raw_scores]
    else:
        norm_scores = [0.0] * len(POSITIVE_CATEGORIES)

    hit_counts = [len(r["hits"][c]) for c in POSITIVE_CATEGORIES]

    has_keyword = 1.0 if total > 0 else 0.0
    is_no_keyword = 1.0 if total == 0 else 0.0

    features = (
        raw_scores
        + norm_scores
        + hit_counts
        + [
            r["best_score"],
            r["second_score"],
            r["margin"],
            r["total_score"],
            r["confidence"],
            has_keyword,
            is_no_keyword
        ]
    )

    return np.array(features, dtype=np.float32)

# ============================================================
# 3. Load dataset
# ============================================================

df = pd.read_csv(DATASET_PATH)

df["category"] = df["category"].astype(str).str.strip()
df = df[df["category"].isin(CATEGORIES)].copy()

df["text_normalized"] = df["text_normalized"].fillna("").astype(str).apply(normalize_arabic)
df = df[df["text_normalized"].str.len() > 0].copy()

train_df = df[df["split"] == "train"].copy().reset_index(drop=True)
val_df = df[df["split"] == "val"].copy().reset_index(drop=True)
test_df = df[df["split"] == "test"].copy().reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

print("\nCategory by split:")
display(pd.crosstab(df["split"], df["category"]))

label_encoder = LabelEncoder()
label_encoder.fit(CATEGORIES)

y_train = label_encoder.transform(train_df["category"])
y_val = label_encoder.transform(val_df["category"])
y_test = label_encoder.transform(test_df["category"])

X_train_text = train_df["text_normalized"].tolist()
X_val_text = val_df["text_normalized"].tolist()
X_test_text = test_df["text_normalized"].tolist()

print("\nLabel mapping:")
for i, c in enumerate(label_encoder.classes_):
    print(i, "->", c)

# ============================================================
# 4. Build features: TF-IDF + MiniLM + Keyword features
# ============================================================

print("\nLoading MiniLM...")
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Encoding MiniLM train...")
X_train_emb = embedder.encode(
    X_train_text,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Encoding MiniLM val...")
X_val_emb = embedder.encode(
    X_val_text,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Encoding MiniLM test...")
X_test_emb = embedder.encode(
    X_test_text,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Building TF-IDF...")
word_tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 6),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_word = word_tfidf.fit_transform(X_train_text)
X_val_word = word_tfidf.transform(X_val_text)
X_test_word = word_tfidf.transform(X_test_text)

X_train_char = char_tfidf.fit_transform(X_train_text)
X_val_char = char_tfidf.transform(X_val_text)
X_test_char = char_tfidf.transform(X_test_text)

print("Building keyword features...")
X_train_kw = np.vstack([keyword_feature_vector(t) for t in X_train_text])
X_val_kw = np.vstack([keyword_feature_vector(t) for t in X_val_text])
X_test_kw = np.vstack([keyword_feature_vector(t) for t in X_test_text])

kw_scaler = StandardScaler()
X_train_kw_scaled = kw_scaler.fit_transform(X_train_kw)
X_val_kw_scaled = kw_scaler.transform(X_val_kw)
X_test_kw_scaled = kw_scaler.transform(X_test_kw)

emb_scaler = StandardScaler()
X_train_emb_scaled = emb_scaler.fit_transform(X_train_emb)
X_val_emb_scaled = emb_scaler.transform(X_val_emb)
X_test_emb_scaled = emb_scaler.transform(X_test_emb)

X_train_all = hstack([
    X_train_word,
    X_train_char,
    csr_matrix(X_train_emb_scaled),
    csr_matrix(X_train_kw_scaled)
]).tocsr()

X_val_all = hstack([
    X_val_word,
    X_val_char,
    csr_matrix(X_val_emb_scaled),
    csr_matrix(X_val_kw_scaled)
]).tocsr()

X_test_all = hstack([
    X_test_word,
    X_test_char,
    csr_matrix(X_test_emb_scaled),
    csr_matrix(X_test_kw_scaled)
]).tocsr()

print("Final feature shape:", X_train_all.shape)

# ============================================================
# 5. Train stronger classifiers
# ============================================================

models = {
    "LogisticRegression_fused": LogisticRegression(
        max_iter=5000,
        C=3.0,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=SEED
    ),

    "LinearSVC_Calibrated_fused": CalibratedClassifierCV(
        estimator=LinearSVC(
            C=1.5,
            class_weight="balanced",
            random_state=SEED
        ),
        cv=3
    ),

    "SGD_LogLoss_fused": SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=1e-5,
        max_iter=3000,
        class_weight="balanced",
        random_state=SEED
    ),
}

def evaluate(model, X, y, split_name):
    y_pred = model.predict(X)

    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X)
        avg_conf = float(np.max(probs, axis=1).mean())
    else:
        probs = None
        avg_conf = None

    acc = accuracy_score(y, y_pred)
    macro_f1 = f1_score(y, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y, y_pred, average="weighted", zero_division=0)
    macro_precision = precision_score(y, y_pred, average="macro", zero_division=0)
    macro_recall = recall_score(y, y_pred, average="macro", zero_division=0)

    return {
        "split": split_name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "avg_confidence": avg_conf,
        "y_pred": y_pred,
        "probs": probs
    }

results = []
trained_models = {}
eval_store = {}

for name, model in models.items():
    print("\n" + "=" * 80)
    print("Training:", name)
    print("=" * 80)

    model.fit(X_train_all, y_train)

    val_eval = evaluate(model, X_val_all, y_val, "val")
    test_eval = evaluate(model, X_test_all, y_test, "test")

    print("\nValidation:")
    print("Accuracy:", round(val_eval["accuracy"], 4))
    print("Macro F1:", round(val_eval["macro_f1"], 4))

    print("\nTest:")
    print("Accuracy:", round(test_eval["accuracy"], 4))
    print("Macro F1:", round(test_eval["macro_f1"], 4))
    print(classification_report(
        y_test,
        test_eval["y_pred"],
        target_names=label_encoder.classes_,
        zero_division=0
    ))

    results.append({
        "model": name,
        "val_accuracy": val_eval["accuracy"],
        "val_macro_f1": val_eval["macro_f1"],
        "val_weighted_f1": val_eval["weighted_f1"],
        "test_accuracy": test_eval["accuracy"],
        "test_macro_f1": test_eval["macro_f1"],
        "test_weighted_f1": test_eval["weighted_f1"],
        "test_macro_precision": test_eval["macro_precision"],
        "test_macro_recall": test_eval["macro_recall"],
        "test_avg_confidence": test_eval["avg_confidence"]
    })

    trained_models[name] = model
    eval_store[name] = {
        "val": val_eval,
        "test": test_eval
    }

results_df = pd.DataFrame(results)
results_df["selection_score"] = (
    0.70 * results_df["val_macro_f1"]
    + 0.30 * results_df["val_accuracy"]
)

results_df = results_df.sort_values(
    by=["selection_score", "val_macro_f1", "val_accuracy"],
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 80)
print("FINAL STAGE 2 V2 MODEL COMPARISON")
print("=" * 80)
display(results_df)

results_df.to_csv(RESULTS_DIR / "stage2_v2_model_comparison.csv", index=False)

best_name = results_df.iloc[0]["model"]
best_model = trained_models[best_name]

print("\n🏆 Best model:", best_name)

# ============================================================
# 6. Feature builder for inference
# ============================================================

def build_single_feature(text):
    text_norm = normalize_arabic(text)

    x_word = word_tfidf.transform([text_norm])
    x_char = char_tfidf.transform([text_norm])

    x_emb = embedder.encode(
        [text_norm],
        normalize_embeddings=True,
        show_progress_bar=False
    )
    x_emb_scaled = emb_scaler.transform(x_emb)

    x_kw = np.vstack([keyword_feature_vector(text_norm)])
    x_kw_scaled = kw_scaler.transform(x_kw)

    x_all = hstack([
        x_word,
        x_char,
        csr_matrix(x_emb_scaled),
        csr_matrix(x_kw_scaled)
    ]).tocsr()

    return text_norm, x_all

# ============================================================
# 7. Improved decision fusion
# ============================================================

def predict_stage2_text_v2(
    text,
    low_conf_threshold=0.55,
    strong_keyword_conf=0.65,
    keyword_close_gap=0.18,
    conflict_uncertain_gap=0.10
):
    text_norm, x = build_single_feature(text)

    probs = best_model.predict_proba(x)[0]
    pred_idx = int(np.argmax(probs))
    ml_category = label_encoder.inverse_transform([pred_idx])[0]
    ml_confidence = float(probs[pred_idx])

    prob_rows = []
    for i, p in enumerate(probs):
        cat = label_encoder.inverse_transform([i])[0]
        prob_rows.append({"category": cat, "probability": float(p)})

    kw = keyword_detector(text_norm)

    final_category = ml_category
    decision = "CATEGORY_MATCH" if ml_category != "غير متعلق بالصيانة" else "IRRELEVANT_TEXT"
    reason = "fused_model_selected"

    if kw["decision"] == "KEYWORD_MATCH":
        kw_cat = kw["category"]
        kw_idx = int(label_encoder.transform([kw_cat])[0])
        kw_model_prob = float(probs[kw_idx])

        # Case 1: keyword and fused model agree
        if kw_cat == ml_category:
            final_category = ml_category
            decision = "CATEGORY_MATCH"
            reason = "fused_model_and_keyword_agree"

        # Case 2: keyword is strong and model is weak or close
        elif (
            kw["confidence"] >= strong_keyword_conf
            and (
                ml_confidence < low_conf_threshold
                or (ml_confidence - kw_model_prob) <= keyword_close_gap
            )
        ):
            final_category = kw_cat
            decision = "CATEGORY_MATCH"
            reason = "strong_keyword_corrected_fused_model"

        # Case 3: strong conflict → do not give wrong final category blindly
        elif abs(ml_confidence - kw_model_prob) <= conflict_uncertain_gap:
            final_category = None
            decision = "UNCERTAIN"
            reason = "keyword_fused_model_conflict_close_scores"

        else:
            final_category = ml_category
            decision = "CATEGORY_MATCH" if ml_category != "غير متعلق بالصيانة" else "IRRELEVANT_TEXT"
            reason = "fused_model_confident_despite_keyword_conflict"

    else:
        if ml_category == "غير متعلق بالصيانة":
            final_category = "غير متعلق بالصيانة"
            decision = "IRRELEVANT_TEXT"
            reason = "no_keyword_fused_model_irrelevant"

        elif kw["decision"] == "NO_KEYWORD" and ml_confidence < low_conf_threshold:
            final_category = None
            decision = "UNCERTAIN"
            reason = "no_keyword_low_model_confidence"

        else:
            final_category = ml_category
            decision = "CATEGORY_MATCH"
            reason = "fused_model_selected_no_strong_keyword"

    return {
        "stage": "Stage 2 Text Classifier v2",
        "input_text": text,
        "normalized_text": text_norm,
        "final_category": final_category,
        "decision": decision,
        "reason": reason,
        "ml_category": ml_category,
        "ml_confidence": ml_confidence,
        "keyword_category": kw["category"],
        "keyword_decision": kw["decision"],
        "keyword_confidence": kw["confidence"],
        "keyword_hits": {k: v for k, v in kw["hits"].items() if len(v) > 0},
        "probabilities": prob_rows,
        "model": best_name
    }

# ============================================================
# 8. Stress test
# ============================================================

stress_examples = [
    ("الحنفية بتسرب ميه", "سباكة"),
    ("الميه بتنقط من تحت الحوض", "سباكة"),
    ("الصرف مسدود والحمام بيطفح", "سباكة"),
    ("السيفون مش شغال", "سباكة"),
    ("فيه بلل تحت الحوض", "سباكة"),
    ("المواسير بايظة", "سباكة"),

    ("النور قاطع في الاوضه", "كهرباء"),
    ("البريزه بتطلع شرار", "كهرباء"),
    ("السلك محروق", "كهرباء"),
    ("المفتاح بيفصل كل شوية", "كهرباء"),
    ("اللمبة مش بتنور", "كهرباء"),
    ("فيشة الكهرباء سخنة", "كهرباء"),

    ("الباب الخشب مش بيقفل", "نجارة"),
    ("الدولاب مفصلته مكسوره", "نجارة"),
    ("الشباك مخلوع", "نجارة"),
    ("مقبض الباب واقع", "نجارة"),
    ("الدرج مش بيفتح", "نجارة"),
    ("الرف الخشب وقع", "نجارة"),

    ("الدهان مقشر في الحيطه", "نقاشة"),
    ("في شرخ كبير في الجدار", "نقاشة"),
    ("السقف فيه رطوبه وبقع", "نقاشة"),
    ("البويه واقعه من الحائط", "نقاشة"),
    ("الحائط فيه تشققات", "نقاشة"),
    ("لون الحائط باهت", "نقاشة"),

    ("القطة واقفة في الشارع", "غير متعلق بالصيانة"),
    ("عايز اصلح الموبايل", "غير متعلق بالصيانة"),
    ("العربية محتاجة غسيل", "غير متعلق بالصيانة"),
    ("الانترنت بطيء والراوتر مش شغال", "غير متعلق بالصيانة"),
    ("الموبايل مش بيشحن", "غير متعلق بالصيانة"),
    ("الكيبورد معلق", "غير متعلق بالصيانة"),
]

stress_rows = []

for text, expected in stress_examples:
    r = predict_stage2_text_v2(text)
    pred = r["final_category"]
    ok = pred == expected

    stress_rows.append({
        "text": text,
        "expected": expected,
        "predicted": pred,
        "decision": r["decision"],
        "reason": r["reason"],
        "ml_category": r["ml_category"],
        "ml_confidence": r["ml_confidence"],
        "keyword_category": r["keyword_category"],
        "keyword_decision": r["keyword_decision"],
        "keyword_confidence": r["keyword_confidence"],
        "ok": ok
    })

stress_df = pd.DataFrame(stress_rows)
stress_df.to_csv(RESULTS_DIR / "stage2_v2_stress_test.csv", index=False)

print("\n" + "=" * 80)
print("STAGE 2 V2 USER-STYLE STRESS TEST")
print("=" * 80)
display(stress_df)

print("\nStress accuracy:", stress_df["ok"].mean())

# ============================================================
# 9. Error analysis on test set
# ============================================================

test_pred = eval_store[best_name]["test"]["y_pred"]

error_df = test_df.copy()
error_df["true_category"] = label_encoder.inverse_transform(y_test)
error_df["pred_category"] = label_encoder.inverse_transform(test_pred)
error_df["correct"] = error_df["true_category"] == error_df["pred_category"]

errors = error_df[~error_df["correct"]].copy()
errors_path = RESULTS_DIR / "stage2_v2_test_errors.csv"
errors.to_csv(errors_path, index=False)

print("\nTest errors:", len(errors))
print("Saved errors to:", errors_path)

if len(errors) > 0:
    display(errors[["text", "true_category", "pred_category", "sample_type", "source"]].head(30))

# ============================================================
# 10. Save artifacts
# ============================================================

joblib.dump(best_model, ARTIFACTS_DIR / "stage2_v2_best_classifier.joblib")
joblib.dump(label_encoder, ARTIFACTS_DIR / "stage2_v2_label_encoder.joblib")
joblib.dump(word_tfidf, ARTIFACTS_DIR / "stage2_v2_word_tfidf.joblib")
joblib.dump(char_tfidf, ARTIFACTS_DIR / "stage2_v2_char_tfidf.joblib")
joblib.dump(emb_scaler, ARTIFACTS_DIR / "stage2_v2_embedding_scaler.joblib")
joblib.dump(kw_scaler, ARTIFACTS_DIR / "stage2_v2_keyword_scaler.joblib")

with open(ARTIFACTS_DIR / "stage2_v2_keywords.json", "w", encoding="utf-8") as f:
    json.dump(CATEGORY_KEYWORDS, f, ensure_ascii=False, indent=2)

info = {
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "best_classifier": best_name,
    "categories": CATEGORIES,
    "dataset_path": str(DATASET_PATH),
    "results_dir": str(RESULTS_DIR),
    "note": "Fused model uses MiniLM embeddings + word TF-IDF + char TF-IDF + keyword score features."
}

with open(ARTIFACTS_DIR / "stage2_v2_info.json", "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)

print("\n✅ Saved artifacts to:", ARTIFACTS_DIR)

# ============================================================
# 11. Gradio
# ============================================================

def gradio_stage2_v2(text):
    if text is None or len(str(text).strip()) == 0:
        return "اكتب وصف المشكلة الأول.", pd.DataFrame(), {}

    r = predict_stage2_text_v2(text)

    probs_df = pd.DataFrame(r["probabilities"])
    if len(probs_df) > 0:
        probs_df = probs_df.sort_values("probability", ascending=False).reset_index(drop=True)

    summary = f"""
### Stage 2 v2 Prediction

**Final category:** {r["final_category"]}
**Decision:** {r["decision"]}
**Reason:** {r["reason"]}

---

**Fused model category:** {r["ml_category"]}
**Fused model confidence:** {r["ml_confidence"]:.4f}

**Keyword category:** {r["keyword_category"]}
**Keyword decision:** {r["keyword_decision"]}
**Keyword confidence:** {r["keyword_confidence"]:.4f}

**Keyword hits:**
`{r["keyword_hits"]}`

**Normalized text:**
`{r["normalized_text"]}`
"""

    raw = {
        "final_category": r["final_category"],
        "decision": r["decision"],
        "reason": r["reason"],
        "ml_category": r["ml_category"],
        "ml_confidence": r["ml_confidence"],
        "keyword_category": r["keyword_category"],
        "keyword_decision": r["keyword_decision"],
        "keyword_confidence": r["keyword_confidence"],
        "keyword_hits": r["keyword_hits"],
        "normalized_text": r["normalized_text"],
        "model": r["model"]
    }

    return summary, probs_df, raw

demo = gr.Interface(
    fn=gradio_stage2_v2,
    inputs=gr.Textbox(
        label="Arabic user description",
        placeholder="مثال: الحنفية بتسرب ميه",
        lines=4
    ),
    outputs=[
        gr.Markdown(label="Prediction Summary"),
        gr.Dataframe(label="Class Probabilities"),
        gr.JSON(label="Raw Output")
    ],
    title="Stage 2 v2 — Improved Arabic Text Classifier",
    description="""
Improved model:
MiniLM + word TF-IDF + char TF-IDF + keyword score features + conservative decision fusion.
""",
    allow_flagging="never"
)

demo.launch(share=True, debug=True)

✅ Using dataset: /content/drive/MyDrive/stage2_unified_clean_dataset.csv
Train: (4131, 20)
Val: (885, 20)
Test: (887, 20)

Category by split:


category,سباكة,غير متعلق بالصيانة,كهرباء,نجارة,نقاشة
split,,,,,
test,188,150,184,186,179
train,876,700,858,865,832
val,188,150,184,185,178



Label mapping:
0 -> سباكة
1 -> غير متعلق بالصيانة
2 -> كهرباء
3 -> نجارة
4 -> نقاشة

Loading MiniLM...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

...

,text,expected,predicted,decision,reason,ml_category,ml_confidence,keyword_category,keyword_decision,keyword_confidence,ok
0,الحنفية بتسرب ميه,سباكة,سباكة,CATEGORY_MATCH,fused_model_and_keyword_agree,سباكة,0.970667,سباكة,KEYWORD_MATCH,1.000000,True
1,الميه بتنقط من تحت الحوض,سباكة,سباكة,CATEGORY_MATCH,fused_model_and_keyword_agree,سباكة,0.984627,سباكة,KEYWORD_MATCH,1.000000,True
2,الصرف مسدود والحمام بيطفح,سباكة,سباكة,CATEGORY_MATCH,fused_model_and_keyword_agree,سباكة,0.981471,سباكة,KEYWORD_MATCH,0.848485,True
3,السيفون مش شغال,سباكة,سباكة,CATEGORY_MATCH,strong_keyword_corrected_fused_model,غير متعلق بالصيانة,0.493715,سباكة,KEYWORD_MATCH,1.000000,True
4,فيه بلل تحت الحوض,سباكة,سباكة,CATEGORY_MATCH,fused_model_and_keyword_agree,سباكة,0.963342,سباكة,KEYWORD_MATCH,1.000000,True
5,المواسير بايظة,سباكة,سباكة,CATEGORY_MATCH,fused_model_and_keyword_agree,سباكة,0.649840,سباكة,KEYWORD_MATCH,1.000000,True
6,النور قاطع في الاوضه,كهرباء,كهرباء,CATEGORY_MATCH,fused_model_and_keyword_agree,كهرباء,0.935749,كهرباء,KEYWORD_MATCH,1.000000,True
7,البريزه بتطلع شرار,كهرباء,كهرباء,CATEGORY_MATCH,fused_model_and_keyword_agree,كهرباء,0.976391,كهرباء,KEYWORD_MATCH,1.000000,True
8,السلك محروق,كهرباء,كهرباء,CATEGORY_MATCH,fused_model_and_keyword_agree,كهرباء,0.953370,كهرباء,KEYWORD_MATCH,1.000000,True
9,المفتاح بيفصل كل شوية,كهرباء,كهرباء,CATEGORY_MATCH,fused_model_and_keyword_agree,كهرباء,0.950436,كهرباء,KEYWORD_MATCH,1.000000,True



Stress accuracy: 1.0

Test errors: 48
Saved errors to: /content/drive/MyDrive/stage2_final_text_v2/results/stage2_v2_test_errors.csv


,text,true_category,pred_category,sample_type,source
0,عبارة عن أجزاء من مسحوق القبو، بما في ذلك الأج...,سباكة,كهرباء,image_generated_description,original_image_caption_cleaned
4,غرفة مفتوحة مع أعمدة معدنية ونوافذ كبيرة تطل ع...,سباكة,نجارة,image_generated_description,original_image_caption_cleaned
9,هناك أضرار في جدار المنزل، حيث تم اكتشاف أن هن...,سباكة,كهرباء,image_generated_description,original_image_caption_cleaned
11,أنت خبير تحليل صور أعطال وصيانة أنت خبير تحليل...,سباكة,كهرباء,image_generated_description,original_image_caption_cleaned
13,، نجد أن هناك أجزاء من الماء والكهرباءءء التي ...,سباكة,كهرباء,image_generated_description,original_image_caption_cleaned
16,، نجد أن هناك أعمدة معدنية كبيرة تقع في الصخور...,سباكة,نجارة,image_generated_description,original_image_caption_cleaned
20,تفاصيل معمارية محددة، حيث تظهر تفاصيل معمارية ...,سباكة,نقاشة,image_generated_description,original_image_caption_cleaned
21,، يتم تجديد وصيانة مبنى مساحته كبيرة يتم استخد...,سباكة,نجارة,image_generated_description,original_image_caption_cleaned
22,هناك أضرار في جدار المطبخ، حيث تم انتشال بعض ا...,سباكة,كهرباء,image_generated_description,original_image_caption_cleaned
23,، غرفة مسورة باللون الذهبي، حيث تم تغيير الألو...,سباكة,نقاشة,image_generated_description,original_image_caption_cleaned



✅ Saved artifacts to: /content/drive/MyDrive/stage2_final_text_v2/artifacts
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fdec567170a1f21404.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fdec567170a1f21404.gradio.live


In [7]:
# ============================================================
# Final Decision Engine
# Stage 1 image_class + Stage 2 text_category
# Output: MATCH / MISMATCH / UNCERTAIN
# ============================================================

!pip install -q gradio timm ultralytics sentence-transformers scikit-learn pandas numpy scipy joblib

import re
import json
import tempfile
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import timm
from torchvision import transforms
from ultralytics import YOLO

from sentence_transformers import SentenceTransformer
from scipy.sparse import hstack, csr_matrix

import gradio as gr

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# ============================================================
# 0. PATHS
# ============================================================

# عدلي المسارات دي لو مختلفة عندك
STAGE1_BEST_DIR_OPTIONS = [
    Path("/content/stage1_visual_dl/best_model"),
    Path("/content/drive/MyDrive/stage1_visual_dl/best_model"),
    Path("/content/drive/MyDrive/stage1_visual_dl_resnet50_best/best_model"),
]

STAGE2_ARTIFACTS_OPTIONS = [
    Path("/content/stage2_final_text_v2/artifacts"),
    Path("/content/drive/MyDrive/stage2_final_text_v2/artifacts"),
    Path("/content/drive/MyDrive/stage2_v2_artifacts"),
]

def find_existing_dir(options, name):
    for p in options:
        if p.exists():
            print(f"✅ Found {name}:", p)
            return p
    raise FileNotFoundError(
        f"Could not find {name}. Tried:\n" + "\n".join(str(p) for p in options)
    )

STAGE1_BEST_DIR = find_existing_dir(STAGE1_BEST_DIR_OPTIONS, "Stage 1 best model dir")
STAGE2_ARTIFACTS_DIR = find_existing_dir(STAGE2_ARTIFACTS_OPTIONS, "Stage 2 artifacts dir")

STAGE1_INFO_PATH = STAGE1_BEST_DIR / "best_model_info.json"
if not STAGE1_INFO_PATH.exists():
    raise FileNotFoundError(f"Stage 1 best_model_info.json not found: {STAGE1_INFO_PATH}")

print("Stage 1 info:", STAGE1_INFO_PATH)
print("Stage 2 artifacts:", STAGE2_ARTIFACTS_DIR)

# ============================================================
# 1. CATEGORY MAPPING
# ============================================================

IMAGE_TO_AR_CATEGORY = {
    "plumbing": "سباكة",
    "electricity": "كهرباء",
    "carpentry": "نجارة",
    "painting": "نقاشة",
    "irrelevant": "irrelevant"
}

AR_TO_IMAGE_CATEGORY = {
    "سباكة": "plumbing",
    "كهرباء": "electricity",
    "نجارة": "carpentry",
    "نقاشة": "painting",
    "غير متعلق بالصيانة": "irrelevant_text"
}

CLASS_DESCRIPTIONS = {
    "plumbing": "سباكة / Plumbing",
    "electricity": "كهرباء / Electricity",
    "carpentry": "نجارة / Carpentry",
    "painting": "نقاشة / Painting",
    "irrelevant": "Irrelevant image / صورة غير متعلقة بالصيانة"
}

# ============================================================
# 2. LOAD STAGE 1 IMAGE MODEL
# ============================================================

with open(STAGE1_INFO_PATH, "r", encoding="utf-8") as f:
    stage1_info = json.load(f)

STAGE1_MODEL_TYPE = stage1_info["type"]
STAGE1_WEIGHTS_PATH = Path(stage1_info["weights_path"])

if not STAGE1_WEIGHTS_PATH.exists():
    candidate = STAGE1_BEST_DIR / "best_model.pt"
    if candidate.exists():
        STAGE1_WEIGHTS_PATH = candidate
    else:
        raise FileNotFoundError(f"Stage 1 weights not found: {stage1_info['weights_path']}")

print("Stage 1 model:", stage1_info["model"])
print("Stage 1 type:", STAGE1_MODEL_TYPE)
print("Stage 1 weights:", STAGE1_WEIGHTS_PATH)

STAGE1_CLASS_NAMES = ["plumbing", "electricity", "carpentry", "painting", "irrelevant"]

if STAGE1_MODEL_TYPE == "YOLO-cls":
    stage1_yolo = YOLO(str(STAGE1_WEIGHTS_PATH))
    stage1_torch = None
    stage1_idx_to_class = None
    stage1_transform = None
else:
    checkpoint = torch.load(STAGE1_WEIGHTS_PATH, map_location=DEVICE)
    model_arch = checkpoint["model_arch"]
    stage1_idx_to_class = {int(k): v for k, v in checkpoint["idx_to_class"].items()}

    stage1_torch = timm.create_model(
        model_arch,
        pretrained=False,
        num_classes=len(STAGE1_CLASS_NAMES)
    ).to(DEVICE)

    stage1_torch.load_state_dict(checkpoint["state_dict"])
    stage1_torch.eval()

    stage1_yolo = None

    stage1_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

def predict_stage1_image(image):
    image = image.convert("RGB")

    if STAGE1_MODEL_TYPE == "YOLO-cls":
        with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
            image.save(tmp.name)
            tmp_path = tmp.name

        result = stage1_yolo(tmp_path, verbose=False)[0]
        probs_tensor = result.probs.data.detach().cpu()
        probs = probs_tensor.numpy()

        names = result.names
        pred_idx = int(result.probs.top1)
        pred_class = names[pred_idx]
        confidence = float(result.probs.top1conf)

        prob_rows = []
        for i, p in enumerate(probs):
            cls = names[i]
            prob_rows.append({
                "image_class": cls,
                "arabic_category": IMAGE_TO_AR_CATEGORY.get(cls, cls),
                "probability": float(p)
            })

    else:
        x = stage1_transform(image).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logits = stage1_torch(x)
            probs_tensor = torch.softmax(logits, dim=1)[0].detach().cpu()
            probs = probs_tensor.numpy()

        pred_idx = int(probs_tensor.argmax().item())
        pred_class = stage1_idx_to_class[pred_idx]
        confidence = float(probs[pred_idx])

        prob_rows = []
        for i, p in enumerate(probs):
            cls = stage1_idx_to_class[i]
            prob_rows.append({
                "image_class": cls,
                "arabic_category": IMAGE_TO_AR_CATEGORY.get(cls, cls),
                "probability": float(p)
            })

    return {
        "stage": "Stage 1 Image Classifier",
        "image_class": pred_class,
        "image_category_ar": IMAGE_TO_AR_CATEGORY.get(pred_class, pred_class),
        "confidence": confidence,
        "probabilities": sorted(prob_rows, key=lambda x: x["probability"], reverse=True)
    }

# ============================================================
# 3. LOAD STAGE 2 TEXT MODEL V2
# ============================================================

stage2_model = joblib.load(STAGE2_ARTIFACTS_DIR / "stage2_v2_best_classifier.joblib")
stage2_label_encoder = joblib.load(STAGE2_ARTIFACTS_DIR / "stage2_v2_label_encoder.joblib")
word_tfidf = joblib.load(STAGE2_ARTIFACTS_DIR / "stage2_v2_word_tfidf.joblib")
char_tfidf = joblib.load(STAGE2_ARTIFACTS_DIR / "stage2_v2_char_tfidf.joblib")
emb_scaler = joblib.load(STAGE2_ARTIFACTS_DIR / "stage2_v2_embedding_scaler.joblib")
kw_scaler = joblib.load(STAGE2_ARTIFACTS_DIR / "stage2_v2_keyword_scaler.joblib")

with open(STAGE2_ARTIFACTS_DIR / "stage2_v2_keywords.json", "r", encoding="utf-8") as f:
    CATEGORY_KEYWORDS = json.load(f)

with open(STAGE2_ARTIFACTS_DIR / "stage2_v2_info.json", "r", encoding="utf-8") as f:
    stage2_info = json.load(f)

EMBEDDING_MODEL_NAME = stage2_info["embedding_model_name"]
print("Loading Stage 2 embedder:", EMBEDDING_MODEL_NAME)
stage2_embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

TEXT_CATEGORIES = list(stage2_label_encoder.classes_)
POSITIVE_TEXT_CATEGORIES = ["سباكة", "كهرباء", "نجارة", "نقاشة"]

print("Stage 2 model:", stage2_info["best_classifier"])
print("Stage 2 classes:", TEXT_CATEGORIES)

def normalize_arabic(text):
    text = "" if pd.isna(text) else str(text)
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ة", "ه").replace("ى", "ي")
    text = text.replace("ؤ", "و").replace("ئ", "ي")
    text = text.replace("گ", "ك")
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)
    text = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

NORMALIZED_KEYWORDS = {
    cat: sorted(set(normalize_arabic(k) for k in kws if normalize_arabic(k)))
    for cat, kws in CATEGORY_KEYWORDS.items()
}

def keyword_detector(text):
    norm = normalize_arabic(text)

    scores = {}
    hits = {}

    for cat, kws in NORMALIZED_KEYWORDS.items():
        score = 0.0
        cat_hits = []

        for kw in kws:
            if kw and kw in norm:
                cat_hits.append(kw)

                if len(kw.split()) >= 2:
                    score += 2.5
                elif len(kw) >= 6:
                    score += 1.5
                elif len(kw) >= 4:
                    score += 1.2
                else:
                    score += 1.0

        scores[cat] = score
        hits[cat] = cat_hits

    best_cat = max(scores, key=scores.get)
    best_score = scores[best_cat]
    total_score = sum(scores.values())

    sorted_scores = sorted(scores.values(), reverse=True)
    second_score = sorted_scores[1] if len(sorted_scores) > 1 else 0.0
    margin = best_score - second_score

    if total_score == 0:
        return {
            "decision": "NO_KEYWORD",
            "category": None,
            "confidence": 0.0,
            "scores": scores,
            "hits": hits,
            "best_score": 0.0,
            "second_score": 0.0,
            "margin": 0.0,
            "total_score": 0.0
        }

    confidence = best_score / total_score

    if best_score >= 1.0 and margin >= 0.5:
        decision = "KEYWORD_MATCH"
    else:
        decision = "KEYWORD_AMBIGUOUS"

    return {
        "decision": decision,
        "category": best_cat,
        "confidence": float(confidence),
        "scores": scores,
        "hits": hits,
        "best_score": float(best_score),
        "second_score": float(second_score),
        "margin": float(margin),
        "total_score": float(total_score)
    }

def keyword_feature_vector(text):
    r = keyword_detector(text)
    scores = r["scores"]

    raw_scores = [scores.get(c, 0.0) for c in POSITIVE_TEXT_CATEGORIES]
    total = sum(raw_scores)

    if total > 0:
        norm_scores = [s / total for s in raw_scores]
    else:
        norm_scores = [0.0] * len(POSITIVE_TEXT_CATEGORIES)

    hit_counts = [len(r["hits"].get(c, [])) for c in POSITIVE_TEXT_CATEGORIES]

    has_keyword = 1.0 if total > 0 else 0.0
    is_no_keyword = 1.0 if total == 0 else 0.0

    features = (
        raw_scores
        + norm_scores
        + hit_counts
        + [
            r["best_score"],
            r["second_score"],
            r["margin"],
            r["total_score"],
            r["confidence"],
            has_keyword,
            is_no_keyword
        ]
    )

    return np.array(features, dtype=np.float32)

def build_stage2_feature(text):
    text_norm = normalize_arabic(text)

    x_word = word_tfidf.transform([text_norm])
    x_char = char_tfidf.transform([text_norm])

    x_emb = stage2_embedder.encode(
        [text_norm],
        normalize_embeddings=True,
        show_progress_bar=False
    )
    x_emb_scaled = emb_scaler.transform(x_emb)

    x_kw = np.vstack([keyword_feature_vector(text_norm)])
    x_kw_scaled = kw_scaler.transform(x_kw)

    x_all = hstack([
        x_word,
        x_char,
        csr_matrix(x_emb_scaled),
        csr_matrix(x_kw_scaled)
    ]).tocsr()

    return text_norm, x_all

def predict_stage2_text(text):
    text_norm, x = build_stage2_feature(text)

    probs = stage2_model.predict_proba(x)[0]
    pred_idx = int(np.argmax(probs))
    ml_category = stage2_label_encoder.inverse_transform([pred_idx])[0]
    ml_confidence = float(probs[pred_idx])

    prob_rows = []
    for i, p in enumerate(probs):
        cat = stage2_label_encoder.inverse_transform([i])[0]
        prob_rows.append({
            "text_category": cat,
            "probability": float(p)
        })

    kw = keyword_detector(text_norm)

    final_category = ml_category
    decision = "CATEGORY_MATCH" if ml_category != "غير متعلق بالصيانة" else "IRRELEVANT_TEXT"
    reason = "stage2_fused_model_selected"

    # Conservative correction
    if kw["decision"] == "KEYWORD_MATCH":
        kw_cat = kw["category"]
        kw_idx = int(stage2_label_encoder.transform([kw_cat])[0])
        kw_model_prob = float(probs[kw_idx])

        if kw_cat == ml_category:
            final_category = ml_category
            decision = "CATEGORY_MATCH"
            reason = "stage2_fused_model_and_keyword_agree"

        elif kw["confidence"] >= 0.65 and (
            ml_confidence < 0.55 or (ml_confidence - kw_model_prob) <= 0.18
        ):
            final_category = kw_cat
            decision = "CATEGORY_MATCH"
            reason = "stage2_strong_keyword_corrected_model"

        elif abs(ml_confidence - kw_model_prob) <= 0.10:
            final_category = None
            decision = "UNCERTAIN"
            reason = "stage2_keyword_model_conflict_close_scores"

    else:
        if ml_category == "غير متعلق بالصيانة":
            final_category = "غير متعلق بالصيانة"
            decision = "IRRELEVANT_TEXT"
            reason = "stage2_no_keyword_model_irrelevant"

        elif kw["decision"] == "NO_KEYWORD" and ml_confidence < 0.55:
            final_category = None
            decision = "UNCERTAIN"
            reason = "stage2_no_keyword_low_confidence"

    return {
        "stage": "Stage 2 Text Classifier",
        "input_text": text,
        "normalized_text": text_norm,
        "text_category": final_category,
        "text_decision": decision,
        "text_reason": reason,
        "ml_category": ml_category,
        "ml_confidence": ml_confidence,
        "keyword_category": kw["category"],
        "keyword_decision": kw["decision"],
        "keyword_confidence": kw["confidence"],
        "keyword_hits": {k: v for k, v in kw["hits"].items() if len(v) > 0},
        "probabilities": sorted(prob_rows, key=lambda x: x["probability"], reverse=True)
    }

# ============================================================
# 4. FINAL DECISION ENGINE
# ============================================================

def final_decision_engine(
    image_result,
    text_result,
    image_conf_threshold=0.60,
    text_conf_threshold=0.55,
    mismatch_uncertain_threshold=0.65
):
    image_class = image_result["image_class"]
    image_category_ar = image_result["image_category_ar"]
    image_conf = image_result["confidence"]

    text_category = text_result["text_category"]
    text_decision = text_result["text_decision"]
    text_conf = text_result["ml_confidence"]

    # 1. Image irrelevant
    # 1. Image irrelevant
    if image_class == "irrelevant":
        return {
            "final_decision": "MISMATCH",
            "reason": "image_is_irrelevant",
            "explanation_ar": "الصورة ليس لها علاقة بالأعطال أو أعمال الصيانة، لذلك لا يمكن مطابقتها مع وصف المستخدم."
        }

    # 2. Text irrelevant
    if text_category == "غير متعلق بالصيانة" or text_decision == "IRRELEVANT_TEXT":
        return {
            "final_decision": "MISMATCH",
            "reason": "text_is_irrelevant",
            "explanation_ar": "وصف المستخدم غير متعلق بالصيانة، لذلك القرار MISMATCH."
        }

    # 3. Text uncertain
    if text_category is None or text_decision == "UNCERTAIN":
        return {
            "final_decision": "UNCERTAIN",
            "reason": "text_category_uncertain",
            "explanation_ar": "النظام غير واثق من تخصص النص، لذلك القرار UNCERTAIN."
        }

    # 4. Low confidence image
    if image_conf < image_conf_threshold:
        return {
            "final_decision": "UNCERTAIN",
            "reason": "low_image_confidence",
            "explanation_ar": f"ثقة موديل الصورة منخفضة ({image_conf:.3f})، لذلك القرار UNCERTAIN."
        }

    # 5. Low confidence text
    if text_conf is not None and text_conf < text_conf_threshold:
        return {
            "final_decision": "UNCERTAIN",
            "reason": "low_text_confidence",
            "explanation_ar": f"ثقة موديل النص منخفضة ({text_conf:.3f})، لذلك القرار UNCERTAIN."
        }

    # 6. Match
    if image_category_ar == text_category:
        return {
            "final_decision": "MATCH",
            "reason": "image_and_text_categories_match",
            "explanation_ar": f"تخصص الصورة ({image_category_ar}) مطابق لتخصص النص ({text_category})، لذلك القرار MATCH."
        }

    # 7. Mismatch or uncertain depending on confidence
    if image_conf < mismatch_uncertain_threshold or (text_conf is not None and text_conf < mismatch_uncertain_threshold):
        return {
            "final_decision": "UNCERTAIN",
            "reason": "category_mismatch_but_confidence_not_high",
            "explanation_ar": (
                f"تخصص الصورة ({image_category_ar}) مختلف عن تخصص النص ({text_category})، "
                f"لكن الثقة ليست عالية كفاية، لذلك القرار UNCERTAIN."
            )
        }

    return {
        "final_decision": "MISMATCH",
        "reason": "image_and_text_categories_do_not_match",
        "explanation_ar": (
            f"تخصص الصورة ({image_category_ar}) مختلف عن تخصص النص ({text_category})، "
            f"لذلك القرار MISMATCH."
        )
    }

def run_full_system(image, user_description):
    if image is None:
        return "من فضلك ارفعي صورة.", pd.DataFrame(), pd.DataFrame(), {}

    if user_description is None or len(str(user_description).strip()) == 0:
        return "من فضلك اكتبي وصف المستخدم.", pd.DataFrame(), pd.DataFrame(), {}

    image_result = predict_stage1_image(image)
    text_result = predict_stage2_text(user_description)
    decision = final_decision_engine(image_result, text_result)

    image_probs_df = pd.DataFrame(image_result["probabilities"])
    text_probs_df = pd.DataFrame(text_result["probabilities"])

    summary = f"""
# Final Decision: **{decision["final_decision"]}**

**Arabic explanation:**
{decision["explanation_ar"]}

---

## Stage 1 — Image Result

**Image class:** {CLASS_DESCRIPTIONS.get(image_result["image_class"], image_result["image_class"])}
**Image category Arabic:** {image_result["image_category_ar"]}
**Image confidence:** {image_result["confidence"]:.4f}

---

## Stage 2 — Text Result

**Text category:** {text_result["text_category"]}
**Text decision:** {text_result["text_decision"]}
**Text reason:** {text_result["text_reason"]}
**Text model category:** {text_result["ml_category"]}
**Text model confidence:** {text_result["ml_confidence"]:.4f}

**Keyword category:** {text_result["keyword_category"]}
**Keyword decision:** {text_result["keyword_decision"]}
**Keyword confidence:** {text_result["keyword_confidence"]:.4f}

**Keyword hits:**
`{text_result["keyword_hits"]}`

---

## Engine Reason

**Reason:** `{decision["reason"]}`
"""

    raw_output = {
        "final_decision": decision,
        "stage1_image": image_result,
        "stage2_text": text_result
    }

    return summary, image_probs_df, text_probs_df, raw_output

# ============================================================
# 5. GRADIO INTERFACE
# ============================================================

demo = gr.Interface(
    fn=run_full_system,
    inputs=[
        gr.Image(type="pil", label="Upload image"),
        gr.Textbox(
            label="Arabic user description",
            placeholder="مثال: الحنفية بتسرب ميه",
            lines=4
        )
    ],
    outputs=[
        gr.Markdown(label="Final Decision"),
        gr.Dataframe(label="Stage 1 Image Probabilities"),
        gr.Dataframe(label="Stage 2 Text Probabilities"),
        gr.JSON(label="Raw Output")
    ],
    title="Final Maintenance Matching System",
    description="""
Stage 1: image_class
Stage 2: text_category
Decision Engine: MATCH / MISMATCH / UNCERTAIN
""",
    allow_flagging="never"
)

demo.launch(share=True, debug=True)

DEVICE: cuda
✅ Found Stage 1 best model dir: /content/drive/MyDrive/stage1_visual_dl/best_model
✅ Found Stage 2 artifacts dir: /content/drive/MyDrive/stage2_final_text_v2/artifacts
Stage 1 info: /content/drive/MyDrive/stage1_visual_dl/best_model/best_model_info.json
Stage 2 artifacts: /content/drive/MyDrive/stage2_final_text_v2/artifacts
Stage 1 model: resnet50
Stage 1 type: timm
Stage 1 weights: /content/drive/MyDrive/stage1_visual_dl/results/resnet50/best_model.pt
Loading Stage 2 embedder: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Stage 2 model: LinearSVC_Calibrated_fused
Stage 2 classes: [np.str_('سباكة'), np.str_('غير متعلق بالصيانة'), np.str_('كهرباء'), np.str_('نجارة'), np.str_('نقاشة')]
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b76636392bb0693b24.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b76636392bb0693b24.gradio.live


In [8]:
# ============================================================
# Build Final Package Structure
# ============================================================

import shutil
from pathlib import Path
import pandas as pd

PACKAGE_DIR = Path("/content/drive/MyDrive/maintenance_matching_system_final")
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

(PACKAGE_DIR / "models" / "stage1").mkdir(parents=True, exist_ok=True)
(PACKAGE_DIR / "models" / "stage2").mkdir(parents=True, exist_ok=True)
(PACKAGE_DIR / "sample_inputs").mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. Artifact paths
# ============================================================

STAGE1_SRC = Path("/content/drive/MyDrive/stage1_visual_dl/best_model")
STAGE2_SRC = Path("/content/drive/MyDrive/stage2_final_text_v2/artifacts")

if not STAGE1_SRC.exists():
    raise FileNotFoundError(f"Stage 1 artifacts not found: {STAGE1_SRC}")

if not STAGE2_SRC.exists():
    raise FileNotFoundError(f"Stage 2 artifacts not found: {STAGE2_SRC}")

# ============================================================
# 2. Copy artifacts
# ============================================================

shutil.copytree(
    STAGE1_SRC,
    PACKAGE_DIR / "models" / "stage1",
    dirs_exist_ok=True
)

shutil.copytree(
    STAGE2_SRC,
    PACKAGE_DIR / "models" / "stage2",
    dirs_exist_ok=True
)

print("✅ Copied Stage 1 artifacts")
print("✅ Copied Stage 2 artifacts")

# ============================================================
# 3. requirements.txt
# ============================================================

requirements = """torch
torchvision
timm
ultralytics
sentence-transformers
scikit-learn
pandas
numpy
scipy
joblib
pillow
gradio
"""

(PACKAGE_DIR / "requirements.txt").write_text(
    requirements.strip(),
    encoding="utf-8"
)

# ============================================================
# 4. README.md
# ============================================================

readme = """# Maintenance Matching System

This system checks whether a maintenance image matches an Arabic user description.

## Pipeline

1. Stage 1 Image Classifier
   - plumbing
   - electricity
   - carpentry
   - painting
   - irrelevant

2. Stage 2 Arabic Text Classifier
   - سباكة
   - كهرباء
   - نجارة
   - نقاشة
   - غير متعلق بالصيانة

3. Decision Engine
   - MATCH
   - MISMATCH
   - UNCERTAIN

## Decision Rules

- If the image is irrelevant:
  الصورة ليس لها علاقة بالأعطال أو أعمال الصيانة.
- If the text is irrelevant:
  وصف المستخدم غير متعلق بالصيانة.
- If image category equals text category:
  MATCH
- If categories are different:
  MISMATCH or UNCERTAIN depending on confidence.

## Run

```bash
pip install -r requirements.txt
python app.py
```
"""

(PACKAGE_DIR / "README.md").write_text(
readme.strip(),
encoding="utf-8"
)

✅ Found Stage 1 artifacts: /content/drive/MyDrive/stage1_visual_dl/best_model
✅ Found Stage 2 artifacts: /content/drive/MyDrive/stage2_final_text_v2/artifacts
✅ Copied Stage 1 to: /content/drive/MyDrive/maintenance_matching_final_package/models/stage1_best_model
✅ Copied Stage 2 to: /content/drive/MyDrive/maintenance_matching_final_package/models/stage2_artifacts


In [9]:
#============================================================
#5. sample_tests.csv
#============================================================

sample_tests = pd.DataFrame([
{
"case_id": 1,
"expected": "MATCH",
"description": "الحنفية بتسرب ميه",
"note": "Use plumbing image"
},
{
"case_id": 2,
"expected": "MISMATCH",
"description": "النور قاطع في الاوضة",
"note": "Use plumbing image"
},
{
"case_id": 3,
"expected": "MISMATCH",
"description": "الباب الخشب مش بيقفل",
"note": "Use irrelevant image"
},
{
"case_id": 4,
"expected": "MISMATCH",
"description": "القطة واقفة في الشارع",
"note": "Use any maintenance image"
},
{
"case_id": 5,
"expected": "MATCH",
"description": "الدهان مقشر في الحيطة",
"note": "Use painting image"
},
])

sample_tests.to_csv(
PACKAGE_DIR / "sample_tests.csv",
index=False,
encoding="utf-8-sig"
)

#============================================================
#6. final_inference.py placeholder
#============================================================

final_inference_note = '''"""
Final inference module.

Paste the tested Decision Engine code here.

This file should include:

load Stage 1 model
load Stage 2 model
predict_stage1_image()
predict_stage2_text()
final_decision_engine()
run_full_system()
"""
'''

(PACKAGE_DIR / "final_inference.py").write_text(
final_inference_note.strip(),
encoding="utf-8"
)

#============================================================
#7. app.py placeholder
#============================================================

app_note = '''"""
Gradio app.

Paste the final Gradio interface code here.

Run:
python app.py
"""
'''

(PACKAGE_DIR / "app.py").write_text(
app_note.strip(),
encoding="utf-8"
)

#============================================================
#8. Zip package
#============================================================

zip_path = shutil.make_archive(
base_name="/content/maintenance_matching_system_final",
format="zip",
root_dir=PACKAGE_DIR
)

print("\n✅ Final package created:")
print(zip_path)

DRIVE_ZIP = Path("/content/drive/MyDrive/maintenance_matching_system_final.zip")
shutil.copy(zip_path, DRIVE_ZIP)

print("\n✅ Copied zip to Drive:")
print(DRIVE_ZIP)


✅ Final package created:
/content/maintenance_matching_system_final.zip

✅ Copied zip to Drive:
/content/drive/MyDrive/maintenance_matching_system_final.zip


In [10]:
import json
import joblib
from pathlib import Path

STAGE2_ARTIFACTS_DIR = Path("/content/drive/MyDrive/stage2_final_text_v2/artifacts")

bundle = {
    "classifier": joblib.load(STAGE2_ARTIFACTS_DIR / "stage2_v2_best_classifier.joblib"),
    "label_encoder": joblib.load(STAGE2_ARTIFACTS_DIR / "stage2_v2_label_encoder.joblib"),
    "word_tfidf": joblib.load(STAGE2_ARTIFACTS_DIR / "stage2_v2_word_tfidf.joblib"),
    "char_tfidf": joblib.load(STAGE2_ARTIFACTS_DIR / "stage2_v2_char_tfidf.joblib"),
    "embedding_scaler": joblib.load(STAGE2_ARTIFACTS_DIR / "stage2_v2_embedding_scaler.joblib"),
    "keyword_scaler": joblib.load(STAGE2_ARTIFACTS_DIR / "stage2_v2_keyword_scaler.joblib"),
}

with open(STAGE2_ARTIFACTS_DIR / "stage2_v2_keywords.json", "r", encoding="utf-8") as f:
    bundle["keywords"] = json.load(f)

with open(STAGE2_ARTIFACTS_DIR / "stage2_v2_info.json", "r", encoding="utf-8") as f:
    bundle["info"] = json.load(f)

bundle["embedding_model_name"] = bundle["info"]["embedding_model_name"]

OUT_PATH = Path("/content/drive/MyDrive/stage2_text_bundle.joblib")
joblib.dump(bundle, OUT_PATH)

print("✅ Stage 2 bundle saved to:")
print(OUT_PATH)

✅ Stage 2 bundle saved to:
/content/drive/MyDrive/stage2_text_bundle.joblib


In [11]:
# ============================================================
# Build Final Team Delivery Folder
# Maintenance Matching System
# Stage 1 Image + Stage 2 Text + Decision Engine
# ============================================================

!pip install -q sentence-transformers joblib pandas

import json
import shutil
import joblib
from pathlib import Path
import pandas as pd
from sentence_transformers import SentenceTransformer

# ============================================================
# 0. CONFIG
# ============================================================

DELIVERY_DIR = Path("/content/maintenance_matching_system_team_delivery")
MODELS_DIR = DELIVERY_DIR / "models"
STAGE1_DST = MODELS_DIR / "stage1"
STAGE2_DST = MODELS_DIR / "stage2"
MINILM_DST = MODELS_DIR / "minilm_model"
SAMPLE_INPUTS_DIR = DELIVERY_DIR / "sample_inputs"

for p in [DELIVERY_DIR, MODELS_DIR, STAGE1_DST, STAGE2_DST, MINILM_DST, SAMPLE_INPUTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# عدلي المسارات دي لو مختلفة عندك
STAGE1_SRC = Path("/content/drive/MyDrive/stage1_visual_dl/best_model")
STAGE2_ARTIFACTS_SRC = Path("/content/drive/MyDrive/stage2_final_text_v2/artifacts")

if not STAGE1_SRC.exists():
    raise FileNotFoundError(f"Stage 1 folder not found: {STAGE1_SRC}")

if not STAGE2_ARTIFACTS_SRC.exists():
    raise FileNotFoundError(f"Stage 2 artifacts folder not found: {STAGE2_ARTIFACTS_SRC}")

# ============================================================
# 1. COPY STAGE 1 ARTIFACTS
# ============================================================

shutil.copytree(STAGE1_SRC, STAGE1_DST, dirs_exist_ok=True)

print("✅ Stage 1 copied to:", STAGE1_DST)

# ============================================================
# 2. CREATE ONE STAGE 2 BUNDLE FILE
# ============================================================

stage2_bundle = {
    "classifier": joblib.load(STAGE2_ARTIFACTS_SRC / "stage2_v2_best_classifier.joblib"),
    "label_encoder": joblib.load(STAGE2_ARTIFACTS_SRC / "stage2_v2_label_encoder.joblib"),
    "word_tfidf": joblib.load(STAGE2_ARTIFACTS_SRC / "stage2_v2_word_tfidf.joblib"),
    "char_tfidf": joblib.load(STAGE2_ARTIFACTS_SRC / "stage2_v2_char_tfidf.joblib"),
    "embedding_scaler": joblib.load(STAGE2_ARTIFACTS_SRC / "stage2_v2_embedding_scaler.joblib"),
    "keyword_scaler": joblib.load(STAGE2_ARTIFACTS_SRC / "stage2_v2_keyword_scaler.joblib"),
}

with open(STAGE2_ARTIFACTS_SRC / "stage2_v2_keywords.json", "r", encoding="utf-8") as f:
    stage2_bundle["keywords"] = json.load(f)

with open(STAGE2_ARTIFACTS_SRC / "stage2_v2_info.json", "r", encoding="utf-8") as f:
    stage2_bundle["info"] = json.load(f)

stage2_bundle["embedding_model_name"] = stage2_bundle["info"]["embedding_model_name"]

STAGE2_BUNDLE_PATH = STAGE2_DST / "stage2_text_bundle.joblib"
joblib.dump(stage2_bundle, STAGE2_BUNDLE_PATH)

print("✅ Stage 2 bundle created:", STAGE2_BUNDLE_PATH)

# ============================================================
# 3. SAVE MINILM LOCALLY FOR OFFLINE / BACKEND USE
# ============================================================

print("Downloading/saving MiniLM locally...")
minilm = SentenceTransformer(stage2_bundle["embedding_model_name"])
minilm.save(str(MINILM_DST))

print("✅ MiniLM saved to:", MINILM_DST)

# ============================================================
# 4. requirements.txt
# ============================================================

requirements = """torch
torchvision
timm
ultralytics
sentence-transformers
scikit-learn
pandas
numpy
scipy
joblib
pillow
gradio
"""

(DELIVERY_DIR / "requirements.txt").write_text(requirements.strip(), encoding="utf-8")

# ============================================================
# 5. final_inference.py
# ============================================================

final_inference_code = r'''
import re
import json
import tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from PIL import Image

import torch
import timm
from torchvision import transforms
from ultralytics import YOLO

from scipy.sparse import hstack, csr_matrix
from sentence_transformers import SentenceTransformer


class MaintenanceMatchingSystem:
    def __init__(self, base_dir=None):
        if base_dir is None:
            base_dir = Path(__file__).resolve().parent
        else:
            base_dir = Path(base_dir)

        self.base_dir = base_dir
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.stage1_dir = self.base_dir / "models" / "stage1"
        self.stage2_bundle_path = self.base_dir / "models" / "stage2" / "stage2_text_bundle.joblib"
        self.minilm_dir = self.base_dir / "models" / "minilm_model"

        self.image_to_ar_category = {
            "plumbing": "سباكة",
            "electricity": "كهرباء",
            "carpentry": "نجارة",
            "painting": "نقاشة",
            "irrelevant": "irrelevant"
        }

        self.class_descriptions = {
            "plumbing": "سباكة / Plumbing",
            "electricity": "كهرباء / Electricity",
            "carpentry": "نجارة / Carpentry",
            "painting": "نقاشة / Painting",
            "irrelevant": "Irrelevant image / صورة ليس لها علاقة بالأعطال"
        }

        self.positive_text_categories = ["سباكة", "كهرباء", "نجارة", "نقاشة"]

        self._load_stage1()
        self._load_stage2()

    def normalize_arabic(self, text):
        text = "" if pd.isna(text) else str(text)
        text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
        text = text.replace("ة", "ه").replace("ى", "ي")
        text = text.replace("ؤ", "و").replace("ئ", "ي")
        text = text.replace("گ", "ك")
        text = re.sub(r"[ًٌٍَُِّْـ]", "", text)
        text = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def _load_stage1(self):
        info_path = self.stage1_dir / "best_model_info.json"
        if not info_path.exists():
            raise FileNotFoundError(f"Stage 1 info file not found: {info_path}")

        with open(info_path, "r", encoding="utf-8") as f:
            self.stage1_info = json.load(f)

        self.stage1_model_type = self.stage1_info["type"]

        weights_path = Path(self.stage1_info.get("weights_path", ""))
        if not weights_path.exists():
            weights_path = self.stage1_dir / "best_model.pt"

        if not weights_path.exists():
            raise FileNotFoundError(f"Stage 1 weights not found: {weights_path}")

        self.stage1_weights_path = weights_path

        if self.stage1_model_type == "YOLO-cls":
            self.stage1_yolo = YOLO(str(weights_path))
            self.stage1_torch = None
            self.stage1_idx_to_class = None
            self.stage1_transform = None
        else:
            checkpoint = torch.load(weights_path, map_location=self.device)
            model_arch = checkpoint["model_arch"]
            self.stage1_idx_to_class = {int(k): v for k, v in checkpoint["idx_to_class"].items()}

            self.stage1_torch = timm.create_model(
                model_arch,
                pretrained=False,
                num_classes=len(self.stage1_idx_to_class)
            ).to(self.device)

            self.stage1_torch.load_state_dict(checkpoint["state_dict"])
            self.stage1_torch.eval()

            self.stage1_yolo = None

            self.stage1_transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
            ])

    def _load_stage2(self):
        if not self.stage2_bundle_path.exists():
            raise FileNotFoundError(f"Stage 2 bundle not found: {self.stage2_bundle_path}")

        bundle = joblib.load(self.stage2_bundle_path)

        self.stage2_model = bundle["classifier"]
        self.stage2_label_encoder = bundle["label_encoder"]
        self.word_tfidf = bundle["word_tfidf"]
        self.char_tfidf = bundle["char_tfidf"]
        self.emb_scaler = bundle["embedding_scaler"]
        self.kw_scaler = bundle["keyword_scaler"]
        self.category_keywords = bundle["keywords"]
        self.stage2_info = bundle["info"]

        if self.minilm_dir.exists():
            self.stage2_embedder = SentenceTransformer(str(self.minilm_dir))
        else:
            self.stage2_embedder = SentenceTransformer(bundle["embedding_model_name"])

        self.normalized_keywords = {
            cat: sorted(set(self.normalize_arabic(k) for k in kws if self.normalize_arabic(k)))
            for cat, kws in self.category_keywords.items()
        }

    def predict_stage1_image(self, image):
        image = image.convert("RGB")

        if self.stage1_model_type == "YOLO-cls":
            with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
                image.save(tmp.name)
                tmp_path = tmp.name

            result = self.stage1_yolo(tmp_path, verbose=False)[0]
            probs_tensor = result.probs.data.detach().cpu()
            probs = probs_tensor.numpy()

            names = result.names
            pred_idx = int(result.probs.top1)
            pred_class = names[pred_idx]
            confidence = float(result.probs.top1conf)

            prob_rows = []
            for i, p in enumerate(probs):
                cls = names[i]
                prob_rows.append({
                    "image_class": cls,
                    "arabic_category": self.image_to_ar_category.get(cls, cls),
                    "probability": float(p)
                })

        else:
            x = self.stage1_transform(image).unsqueeze(0).to(self.device)

            with torch.no_grad():
                logits = self.stage1_torch(x)
                probs_tensor = torch.softmax(logits, dim=1)[0].detach().cpu()
                probs = probs_tensor.numpy()

            pred_idx = int(probs_tensor.argmax().item())
            pred_class = self.stage1_idx_to_class[pred_idx]
            confidence = float(probs[pred_idx])

            prob_rows = []
            for i, p in enumerate(probs):
                cls = self.stage1_idx_to_class[i]
                prob_rows.append({
                    "image_class": cls,
                    "arabic_category": self.image_to_ar_category.get(cls, cls),
                    "probability": float(p)
                })

        return {
            "stage": "Stage 1 Image Classifier",
            "image_class": pred_class,
            "image_category_ar": self.image_to_ar_category.get(pred_class, pred_class),
            "confidence": confidence,
            "probabilities": sorted(prob_rows, key=lambda x: x["probability"], reverse=True)
        }

    def keyword_detector(self, text):
        norm = self.normalize_arabic(text)
        scores = {}
        hits = {}

        for cat, kws in self.normalized_keywords.items():
            score = 0.0
            cat_hits = []

            for kw in kws:
                if kw and kw in norm:
                    cat_hits.append(kw)

                    if len(kw.split()) >= 2:
                        score += 2.5
                    elif len(kw) >= 6:
                        score += 1.5
                    elif len(kw) >= 4:
                        score += 1.2
                    else:
                        score += 1.0

            scores[cat] = score
            hits[cat] = cat_hits

        best_cat = max(scores, key=scores.get)
        best_score = scores[best_cat]
        total_score = sum(scores.values())

        sorted_scores = sorted(scores.values(), reverse=True)
        second_score = sorted_scores[1] if len(sorted_scores) > 1 else 0.0
        margin = best_score - second_score

        if total_score == 0:
            return {
                "decision": "NO_KEYWORD",
                "category": None,
                "confidence": 0.0,
                "scores": scores,
                "hits": hits,
                "best_score": 0.0,
                "second_score": 0.0,
                "margin": 0.0,
                "total_score": 0.0
            }

        confidence = best_score / total_score

        if best_score >= 1.0 and margin >= 0.5:
            decision = "KEYWORD_MATCH"
        else:
            decision = "KEYWORD_AMBIGUOUS"

        return {
            "decision": decision,
            "category": best_cat,
            "confidence": float(confidence),
            "scores": scores,
            "hits": hits,
            "best_score": float(best_score),
            "second_score": float(second_score),
            "margin": float(margin),
            "total_score": float(total_score)
        }

    def keyword_feature_vector(self, text):
        r = self.keyword_detector(text)
        scores = r["scores"]

        raw_scores = [scores.get(c, 0.0) for c in self.positive_text_categories]
        total = sum(raw_scores)

        if total > 0:
            norm_scores = [s / total for s in raw_scores]
        else:
            norm_scores = [0.0] * len(self.positive_text_categories)

        hit_counts = [len(r["hits"].get(c, [])) for c in self.positive_text_categories]

        has_keyword = 1.0 if total > 0 else 0.0
        is_no_keyword = 1.0 if total == 0 else 0.0

        features = (
            raw_scores
            + norm_scores
            + hit_counts
            + [
                r["best_score"],
                r["second_score"],
                r["margin"],
                r["total_score"],
                r["confidence"],
                has_keyword,
                is_no_keyword
            ]
        )

        return np.array(features, dtype=np.float32)

    def build_stage2_feature(self, text):
        text_norm = self.normalize_arabic(text)

        x_word = self.word_tfidf.transform([text_norm])
        x_char = self.char_tfidf.transform([text_norm])

        x_emb = self.stage2_embedder.encode(
            [text_norm],
            normalize_embeddings=True,
            show_progress_bar=False
        )
        x_emb_scaled = self.emb_scaler.transform(x_emb)

        x_kw = np.vstack([self.keyword_feature_vector(text_norm)])
        x_kw_scaled = self.kw_scaler.transform(x_kw)

        x_all = hstack([
            x_word,
            x_char,
            csr_matrix(x_emb_scaled),
            csr_matrix(x_kw_scaled)
        ]).tocsr()

        return text_norm, x_all

    def predict_stage2_text(self, text):
        text_norm, x = self.build_stage2_feature(text)

        probs = self.stage2_model.predict_proba(x)[0]
        pred_idx = int(np.argmax(probs))
        ml_category = self.stage2_label_encoder.inverse_transform([pred_idx])[0]
        ml_confidence = float(probs[pred_idx])

        prob_rows = []
        for i, p in enumerate(probs):
            cat = self.stage2_label_encoder.inverse_transform([i])[0]
            prob_rows.append({
                "text_category": cat,
                "probability": float(p)
            })

        kw = self.keyword_detector(text_norm)

        final_category = ml_category
        decision = "CATEGORY_MATCH" if ml_category != "غير متعلق بالصيانة" else "IRRELEVANT_TEXT"
        reason = "stage2_fused_model_selected"

        if kw["decision"] == "KEYWORD_MATCH":
            kw_cat = kw["category"]
            kw_idx = int(self.stage2_label_encoder.transform([kw_cat])[0])
            kw_model_prob = float(probs[kw_idx])

            if kw_cat == ml_category:
                final_category = ml_category
                decision = "CATEGORY_MATCH"
                reason = "stage2_fused_model_and_keyword_agree"

            elif kw["confidence"] >= 0.65 and (
                ml_confidence < 0.55 or (ml_confidence - kw_model_prob) <= 0.18
            ):
                final_category = kw_cat
                decision = "CATEGORY_MATCH"
                reason = "stage2_strong_keyword_corrected_model"

            elif abs(ml_confidence - kw_model_prob) <= 0.10:
                final_category = None
                decision = "UNCERTAIN"
                reason = "stage2_keyword_model_conflict_close_scores"

        else:
            if ml_category == "غير متعلق بالصيانة":
                final_category = "غير متعلق بالصيانة"
                decision = "IRRELEVANT_TEXT"
                reason = "stage2_no_keyword_model_irrelevant"

            elif kw["decision"] == "NO_KEYWORD" and ml_confidence < 0.55:
                final_category = None
                decision = "UNCERTAIN"
                reason = "stage2_no_keyword_low_confidence"

        return {
            "stage": "Stage 2 Text Classifier",
            "input_text": text,
            "normalized_text": text_norm,
            "text_category": final_category,
            "text_decision": decision,
            "text_reason": reason,
            "ml_category": ml_category,
            "ml_confidence": ml_confidence,
            "keyword_category": kw["category"],
            "keyword_decision": kw["decision"],
            "keyword_confidence": kw["confidence"],
            "keyword_hits": {k: v for k, v in kw["hits"].items() if len(v) > 0},
            "probabilities": sorted(prob_rows, key=lambda x: x["probability"], reverse=True)
        }

    def final_decision_engine(
        self,
        image_result,
        text_result,
        image_conf_threshold=0.60,
        text_conf_threshold=0.55,
        mismatch_uncertain_threshold=0.65
    ):
        image_class = image_result["image_class"]
        image_category_ar = image_result["image_category_ar"]
        image_conf = image_result["confidence"]

        text_category = text_result["text_category"]
        text_decision = text_result["text_decision"]
        text_conf = text_result["ml_confidence"]

        if image_class == "irrelevant":
            return {
                "final_decision": "MISMATCH",
                "reason": "image_is_irrelevant",
                "explanation_ar": "الصورة ليس لها علاقة بالأعطال أو أعمال الصيانة، لذلك لا يمكن مطابقتها مع وصف المستخدم."
            }

        if text_category == "غير متعلق بالصيانة" or text_decision == "IRRELEVANT_TEXT":
            return {
                "final_decision": "MISMATCH",
                "reason": "text_is_irrelevant",
                "explanation_ar": "وصف المستخدم غير متعلق بالصيانة، لذلك القرار MISMATCH."
            }

        if text_category is None or text_decision == "UNCERTAIN":
            return {
                "final_decision": "UNCERTAIN",
                "reason": "text_category_uncertain",
                "explanation_ar": "النظام غير واثق من تخصص النص، لذلك القرار UNCERTAIN."
            }

        if image_conf < image_conf_threshold:
            return {
                "final_decision": "UNCERTAIN",
                "reason": "low_image_confidence",
                "explanation_ar": f"ثقة موديل الصورة منخفضة ({image_conf:.3f})، لذلك القرار UNCERTAIN."
            }

        if text_conf is not None and text_conf < text_conf_threshold:
            return {
                "final_decision": "UNCERTAIN",
                "reason": "low_text_confidence",
                "explanation_ar": f"ثقة موديل النص منخفضة ({text_conf:.3f})، لذلك القرار UNCERTAIN."
            }

        if image_category_ar == text_category:
            return {
                "final_decision": "MATCH",
                "reason": "image_and_text_categories_match",
                "explanation_ar": f"تخصص الصورة ({image_category_ar}) مطابق لتخصص النص ({text_category})، لذلك القرار MATCH."
            }

        if image_conf < mismatch_uncertain_threshold or (text_conf is not None and text_conf < mismatch_uncertain_threshold):
            return {
                "final_decision": "UNCERTAIN",
                "reason": "category_mismatch_but_confidence_not_high",
                "explanation_ar": (
                    f"تخصص الصورة ({image_category_ar}) مختلف عن تخصص النص ({text_category})، "
                    f"لكن الثقة ليست عالية كفاية، لذلك القرار UNCERTAIN."
                )
            }

        return {
            "final_decision": "MISMATCH",
            "reason": "image_and_text_categories_do_not_match",
            "explanation_ar": (
                f"تخصص الصورة ({image_category_ar}) مختلف عن تخصص النص ({text_category})، "
                f"لذلك القرار MISMATCH."
            )
        }

    def predict(self, image, user_description):
        image_result = self.predict_stage1_image(image)
        text_result = self.predict_stage2_text(user_description)
        decision = self.final_decision_engine(image_result, text_result)

        return {
            "final_decision": decision,
            "stage1_image": image_result,
            "stage2_text": text_result
        }
'''

(DELIVERY_DIR / "final_inference.py").write_text(final_inference_code.strip(), encoding="utf-8")

# ============================================================
# 6. app.py
# ============================================================

app_code = r'''
import pandas as pd
import gradio as gr
from final_inference import MaintenanceMatchingSystem

system = MaintenanceMatchingSystem()

def run_app(image, user_description):
    if image is None:
        return "من فضلك ارفع صورة.", pd.DataFrame(), pd.DataFrame(), {}

    if user_description is None or len(str(user_description).strip()) == 0:
        return "من فضلك اكتب وصف المستخدم.", pd.DataFrame(), pd.DataFrame(), {}

    output = system.predict(image, user_description)

    decision = output["final_decision"]
    image_result = output["stage1_image"]
    text_result = output["stage2_text"]

    image_probs_df = pd.DataFrame(image_result["probabilities"])
    text_probs_df = pd.DataFrame(text_result["probabilities"])

    summary = f"""
# Final Decision: **{decision["final_decision"]}**

**Arabic explanation:**
{decision["explanation_ar"]}

---

## Stage 1 — Image Result

**Image class:** {image_result["image_class"]}
**Image category Arabic:** {image_result["image_category_ar"]}
**Image confidence:** {image_result["confidence"]:.4f}

---

## Stage 2 — Text Result

**Text category:** {text_result["text_category"]}
**Text decision:** {text_result["text_decision"]}
**Text reason:** {text_result["text_reason"]}
**Text model category:** {text_result["ml_category"]}
**Text model confidence:** {text_result["ml_confidence"]:.4f}

**Keyword category:** {text_result["keyword_category"]}
**Keyword decision:** {text_result["keyword_decision"]}
**Keyword confidence:** {text_result["keyword_confidence"]:.4f}

**Keyword hits:**
`{text_result["keyword_hits"]}`

---

## Engine Reason

**Reason:** `{decision["reason"]}`
"""

    return summary, image_probs_df, text_probs_df, output


demo = gr.Interface(
    fn=run_app,
    inputs=[
        gr.Image(type="pil", label="Upload image"),
        gr.Textbox(
            label="Arabic user description",
            placeholder="مثال: الحنفية بتسرب ميه",
            lines=4
        )
    ],
    outputs=[
        gr.Markdown(label="Final Decision"),
        gr.Dataframe(label="Stage 1 Image Probabilities"),
        gr.Dataframe(label="Stage 2 Text Probabilities"),
        gr.JSON(label="Raw Output")
    ],
    title="Maintenance Matching System",
    description="Stage 1 Image Classifier + Stage 2 Arabic Text Classifier + Decision Engine",
    allow_flagging="never"
)

if __name__ == "__main__":
    demo.launch(share=False, debug=True)
'''

(DELIVERY_DIR / "app.py").write_text(app_code.strip(), encoding="utf-8")

# ============================================================
# 7. sample_tests.csv
# ============================================================

sample_tests = pd.DataFrame([
    {
        "case_id": 1,
        "expected": "MATCH",
        "description": "الحنفية بتسرب ميه",
        "note": "Use plumbing image"
    },
    {
        "case_id": 2,
        "expected": "MISMATCH",
        "description": "النور قاطع في الاوضة",
        "note": "Use plumbing image"
    },
    {
        "case_id": 3,
        "expected": "MISMATCH",
        "description": "الباب الخشب مش بيقفل",
        "note": "Use irrelevant image"
    },
    {
        "case_id": 4,
        "expected": "MISMATCH",
        "description": "القطة واقفة في الشارع",
        "note": "Use any maintenance image"
    },
    {
        "case_id": 5,
        "expected": "MATCH",
        "description": "الدهان مقشر في الحيطة",
        "note": "Use painting image"
    },
])

sample_tests.to_csv(
    DELIVERY_DIR / "sample_tests.csv",
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 8. README.md
# ============================================================

readme = """# Maintenance Matching System

This package contains the final maintenance matching system.

## Task

The system checks whether an uploaded maintenance image matches the Arabic user description.

## Pipeline

1. Stage 1 Image Classifier
   - plumbing
   - electricity
   - carpentry
   - painting
   - irrelevant

2. Stage 2 Arabic Text Classifier
   - سباكة
   - كهرباء
   - نجارة
   - نقاشة
   - غير متعلق بالصيانة

3. Decision Engine
   - MATCH
   - MISMATCH
   - UNCERTAIN

## Folder Structure

```text
maintenance_matching_system_team_delivery/
│
├── app.py
├── final_inference.py
├── requirements.txt
├── README.md
├── sample_tests.csv
│
├── models/
│   ├── stage1/
│   │   ├── best_model.pt
│   │   └── best_model_info.json
│   │
│   ├── stage2/
│   │   └── stage2_text_bundle.joblib
│   │
│   └── minilm_model/
│
└── sample_inputs
"""
(DELIVERY_DIR / "README.md").write_text(readme.strip(), encoding="utf-8")

#============================================================
#9. Zip final folder
#============================================================

ZIP_PATH = shutil.make_archive(
base_name="/content/maintenance_matching_system_team_delivery",
format="zip",
root_dir=DELIVERY_DIR
)

DRIVE_ZIP = Path("/content/drive/MyDrive/maintenance_matching_system_team_delivery.zip")
shutil.copy(ZIP_PATH, DRIVE_ZIP)

print("\n============================================================")
print("✅ FINAL TEAM DELIVERY READY")
print("============================================================")
print("Folder:", DELIVERY_DIR)
print("Zip:", ZIP_PATH)
print("Drive Zip:", DRIVE_ZIP)
print("\nSend this zip to your backend/team:")
print(DRIVE_ZIP)

✅ Stage 1 copied to: /content/maintenance_matching_system_team_delivery/models/stage1
✅ Stage 2 bundle created: /content/maintenance_matching_system_team_delivery/models/stage2/stage2_text_bundle.joblib
Downloading/saving MiniLM locally...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ MiniLM saved to: /content/maintenance_matching_system_team_delivery/models/minilm_model

✅ FINAL TEAM DELIVERY READY
Folder: /content/maintenance_matching_system_team_delivery
Zip: /content/maintenance_matching_system_team_delivery.zip
Drive Zip: /content/drive/MyDrive/maintenance_matching_system_team_delivery.zip

Send this zip to your backend/team:
/content/drive/MyDrive/maintenance_matching_system_team_delivery.zip
